# 4 Neural Networks

Our goal is to make 4 neural nets:
1. We have already made the first, out encoder which is an MLP that takes us to 512 ChemNet space
2. An MLP that will take spectra and output toxicity. We also want to make an RF that will do the same
3. An MLP that will take ChemNet embeddings and output toxicity, again we already have a RF that does this, which limited success for unknown chemical compounds.
4. The most complicated, a condiotnal encoder that will intake Spectra and output 516 space, which one hot encoded EPA levels as additinal conditions. This has many further considerations since it will need to peicewise loss function. 


Another condsideration as we start getting these NNs up and running is what problem we want to tackle, multi-class or binary classification and regression are the main two. Additionally we have been using EPA LD50 toxicity levels but the Georgia Institute of Technology indicates that "A substance is considered extremely toxic if it has an LD50 of less than 5 mgs/kg of animal body weight." This could be another bench mark of binary classification, classifying only "extremely toxic" chemicals with LD50 socres below 5. The primary problem that this poses is that there are just VERY few chemicals that fit this description, making major balance issues, compounding the existing problem, as even with an EPA split we see relatively few chemcals of even EPA level 1.

However EPA levels are revealing as "it is Highly toxic if it has an LD50 of between 5 and 50 mg/kg of animal body weight to a human, this would be about a teaspoon." And 50 is the EPA level 1 benchmark, so could also be used as a benchmark for binary classification, however the balance issues would still persist, and they would be inconsistent across groups. 

# General: Functions, Data Upload etc.

## Imports and Globally Used Functions

In [ ]:
# Package imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import functions_enc as f

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor



In [ ]:
# Spectrum string to dataframe function
def spectrum_string_to_dataframe(df, spectrum_col, smiles_col):
    """
    Converts a DataFrame with a spectrum column (string of 'x:y' pairs) into a matrix
    where columns are unique x values, rows are spectra (even for duplicate SMILES), and values are y (intensity).
    The index will match the original DataFrame.
    """
    # Collect all unique x values (m/z)
    x_values_set = set()
    spectra_list = []
    for idx, row in df.iterrows():
        spectrum = row[spectrum_col]
        pairs = spectrum.split()
        xys = []
        for pair in pairs:
            try:
                x, y = pair.split(":") # Split into x and y
                #x = float(x.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                #y = float(y.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                xys.append((x, y))
                x_values_set.add(x)
            except Exception:
                continue
        spectra_list.append((row[smiles_col], dict(xys)))
    x_values = sorted(x_values_set) # Sort the x values to maintain order
    
    # Build the matrix
    matrix = []
    smiles_list = []
    for smiles, xy_dict in spectra_list:
        row = [xy_dict.get(x, 0.0) for x in x_values]
        matrix.append(row)
        smiles_list.append(smiles)
    df_matrix = pd.DataFrame(matrix, columns=[x for x in x_values]) # columns=[f"mz_{x}" for x in x_values]) to make stings
    df_matrix.insert(0, smiles_col, smiles_list)
    df_matrix.index = df.index  # preserve original row order/index
    return df_matrix

In [ ]:
# Binning functions
# Binning the spectra data 
def bin_spectra_by_integer_mz(df):
    """
    Bins the spectra data by rounding m/z (column names) to the nearest integer,
    then sums intensities for duplicate integer bins.
    Assumes first column is SMILES, rest are m/z columns (floats).
    """
    smiles_col = df.columns[0]
    spectra = df.iloc[:, 1:]
    # Map each column to its integer bin
    int_mz = [int(round(float(c))) for c in spectra.columns]
    spectra.columns = int_mz
    # Group columns by integer m/z and sum
    binned = spectra.groupby(level=0, axis=1).sum()
    # Add the SMILES column back
    binned.insert(0, smiles_col, df[smiles_col])
    return binned

# Fill in the missing integer columns 
def fill_missing_integer_columns(df):
    """
    Ensures all integer columns from 1 to the maximum are present in the DataFrame.
    Missing columns are filled with zeros.
    Assumes the first column is the label (e.g., SMILES).
    """
    # Get the integer columns (skip the first column)
    int_cols = [col for col in df.columns[1:] if isinstance(col, int)]
    #min_col = min(int_cols)
    max_col = max(int_cols)
    all_int_cols = list(range(1, max_col + 1))
    # Find missing columns
    missing_cols = set(all_int_cols) - set(int_cols)
    # Add missing columns with zeros
    for col in missing_cols:
        df[col] = 0.0
    # Reorder columns: first column, then sorted integer columns
    ordered_cols = [df.columns[0]] + sorted(all_int_cols)
    df = df[ordered_cols]
    return df

In [ ]:
# Cate's smiles to ChemNet embedding code
def get_chemnet_emb_from_smiles(smiles_list):
    """
    Get ChemNet embeddings from a list of SMILES strings.

    Parameters:
    smiles_list (list): List of SMILES strings.

    Returns:
    dict: A dictionary mapping each SMILES string to its corresponding ChemNet embedding.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fcd = FCD(device, n_jobs=1)
    
    smiles_emb_dict = {}

    for smiles in smiles_list:
        try:
            emb = fcd.get_predictions([smiles])[0]
            smiles_emb_dict[smiles] = list(emb)
        except KeyError as e:
            if e == 'PropertyTable':
                smiles_emb_dict[smiles] = 'unknown'

    return smiles_emb_dict

In [ ]:
# Define a function to assign EPA levels
def assign_epa_level(response):
    if response <= 50:
        return "EPA_level_1"
    elif response <= 500:
        return "EPA_level_2"
    elif response <= 5000:
        return "EPA_level_3"
    else:
        return "EPA_level_4"

In [ ]:
# Add the 'Response' and 'log_response' columns THIS IS A UNIVERSAL FUNCTION
def add_response_and_log_response(spectra_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds 'Response' and 'log_response' columns to spectra_df by mapping from original_df using the SMILES column.
    """
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    spectra_df['Response'] = spectra_df[smiles_col].map(smiles_to_response)
    spectra_df['log_response'] = np.log(spectra_df['Response'])
    return spectra_df

In [ ]:
# Add the 'Response' column using the SMILES as an identifier
def add_response_column_to_spectra(spectra_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds a 'Response' column to spectra_df by mapping from original_df using the SMILES column.
    """
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    spectra_df['Response'] = spectra_df[smiles_col].map(smiles_to_response)
    return spectra_df

## Initial Data upload and processing

In [ ]:
# The 5/30 dataset with rat based toxicity data and groups
df3 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data3.csv")
print(df3.shape)
df3.head()

# Uniformity of ionization model labels
print(df3["Ionization_Mode"].unique())
df3["Ionization_Mode"] = df3["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df3["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df3 = df3[df3["Ionization_Mode"] != "'NaN'"]
print(df3["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df3
df3 = df3.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df3["SMILES_spectra"] = df3["SMILES_spectra"].str.replace("'", "")

# This will give us the subsets with all of the relevant information
df3_QQpos = df3[df3['Group'] == 'Q-Orbitrap-positive'] # 1307
df3_QQneg = df3[df3['Group'] == 'Q-Orbitrap-negative'] # 756
df3_QTOFpos = df3[df3['Group'] == 'Q-TOF-positive'] # 736  
df3_LTQOpos = df3[df3['Group'] == 'LTQ-Orbitrap-positive'] # 481 


# Now since I have saved files of the binned specrtra I can simply call those
df3_QQpos_spectra = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv")
df3_QQneg_spectra = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQneg_spectra.csv")
df3_LTQOpos_spectra = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_LTQOpos_spectra.csv")
df3_QTOFpos_spectra = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QTOFpos_spectra.csv")


In [ ]:
# Counts of extrmemly toxic chemicals
num_extremely_toxic = (df3_QQpos['Response'] < 20).sum()
print(f"Number of SMILES with Response < 10: {num_extremely_toxic}")

num_extremely_toxic = (df3_QQneg['Response'] < 20).sum()
print(f"Number of SMILES with Response < 10: {num_extremely_toxic}")

num_extremely_toxic = (df3_QTOFpos['Response'] < 20).sum()
print(f"Number of SMILES with Response < 10: {num_extremely_toxic}")

num_extremely_toxic = (df3_LTQOpos['Response'] < 20).sum()
print(f"Number of SMILES with Response < 10: {num_extremely_toxic}")

# Spectra --> ChemNet Encoder

### Functions

In [ ]:
# This is our default function, the one we use to prep the data for the encoder that takes us from spectra to ChemNet encodings 
def create_dataset_tensors(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):
    """
    Create tensors from the provided spectra dataset and embedding DataFrame.

    Parameters:
    ----------
    spectra_dataset : pd.DataFrame
        DataFrame containing spectral data and chemical labels. Assumes specific 
        columns for processing based on the `carl` flag.

    embedding_df : pd.DataFrame
        DataFrame containing embeddings for chemicals, with 'Embedding Floats' 
        column corresponding to ChemNet embeddings.

    device : torch.device
        The device (CPU or GPU) on which to store the tensors.

    carl : bool, optional
        If True, processes the dataset assuming it has a different structure 
        (specifically without an 'Unnamed: 0' column). Default is False.

    Returns:
    -------
    tuple
        A tuple containing:
        - embeddings_tensor (torch.Tensor): Tensor of true embeddings for the chemicals.
        - spectra_tensor (torch.Tensor): Tensor of spectral data.
        - chem_encodings_tensor (torch.Tensor): Tensor of chemical name encodings.
        - spectra_indices_tensor (torch.Tensor): Tensor of indices corresponding to the spectra.
    """
    spectra = spectra_dataset.iloc[:,1:-1]

    #chem_encodings = spectra_dataset.iloc[:,-8:]

    # create tensors of spectra, true embeddings, and chemical name encodings for train and val
    chem_labels = list(spectra_dataset['SMILES_spectra'])
    embeddings_tensor = torch.Tensor([embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) for chem_name in chem_labels]).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    #chem_encodings_tensor = torch.Tensor(chem_encodings.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return embeddings_tensor, spectra_tensor, spectra_indices_tensor #, chem_encodings_tensor

### Architecture

In [ ]:
batch_size = 64
epochs=500
lr=0.0001
criterion=nn.MSELoss()
output_size = 512
num_layers = 5

#%%
# Encoder architecture (With Validation Set)
class Encoder(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model_encoder(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_embeddings, _ in train_data:
            batch = batch.to(device)
            true_embeddings = true_embeddings.to(device)

            optimizer.zero_grad()
            batch_predicted_embeddings = model(batch)
            loss = criterion(batch_predicted_embeddings, true_embeddings) # loss1 (embedding loss) and loss2 (toxicity loss)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_embeddings, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_embeddings.to(device)

                val_batch_predicted_embeddings = model(val_batch)

                val_loss = criterion(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model
#%%


### QQpos Training

In [ ]:
# Training and validation dataset split 
df3_QQpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")
# Generalize the syntax
dataset =  df3_QQpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQpos_copy = train_data.copy()
test_data_df3_QQpos_copy = test_data.copy()

# Save under correct names
train_data_df3_QQpos = train_data
test_data_df3_QQpos = test_data
# Load val_data
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 

In [ ]:
# Remove the specified SMILES from the train and test sets,
# and collect them into a new DataFrame called super_testing_df3_QQpos

smiles_to_remove = ['CCCSP(=O)(OCC)SCCC', 'C#CCN(C)Cc1ccccc1', 'C1=CC(=CC=N1)C1=CC=NC=C1', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'C(NC1=NC=NC2=C1N=CN2C1CCCCO1)C1=CC=CC=C1', 'CC(C)N(CC1=CC=CC=C1)C(=O)C(C)(C)C', 'NC(=S)NC1=CC=CC=C1', 
                    'CC(C)(C)C1=CC(=CC(=C1O)C(C)(C)C)[N+]([O-])=O', 'CC(C)(C)C1=CC=C(C=C1)C(O)=O', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'CC(=C)C(=O)NC1=CC=C(Cl)C(Cl)=C1', 'CC(C)(C1=CC(Cl)=C(O)C(Cl)=C1)C1=CC(Cl)=C(O)C(Cl)=C1', 
                    'CCC(=O)N(C1CCN(CCC2=CC=CC=C2)CC1)C1=CC=CC=C1', 'CC(=O)N1CCN(CC1)C1=CC=C(OC[C@H]2CO[C@@](CN3C=CN=C3)(O2)C2=CC=C(Cl)C=C2Cl)C=C1', 
                    'CC(C)(C)C(=O)C(OC1=CC=C(Cl)C=C1)N1C=CN=C1', 'C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1', 'CC(=O)OCC[N+](C)(C)C', 
                    'CC(=O)NC1=CC=C(C=C1)S(=O)(=O)NC1=NOC(C)=C1', 'C1CCC(CC1)NC1CCCCC1', 'CC(CN1CCN(CCOCCO)CC1)CN1C2=CC=CC=C2SC2=CC=CC=C12', 
                    'CC(=O)OCC(COC(C)=O)OC(C)=O', 'CC(C)N(C(=O)CCl)c1ccccc1', 'CC1=C(C)N=C(NS(=O)(=O)C2=CC=C(N)C=C2)O1']

# Combine train and test to extract all rows with those SMILES
combined_df3_QQpos = pd.concat([train_data_df3_QQpos, test_data_df3_QQpos], ignore_index=True)
super_testing_df3_QQpos = combined_df3_QQpos[combined_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

# Remove those SMILES from train and test sets
train_data_df3_QQpos = train_data_df3_QQpos[~train_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()
test_data_df3_QQpos = test_data_df3_QQpos[~test_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

print("After removal, train shape:", train_data_df3_QQpos.shape)
print("After removal, test shape:", test_data_df3_QQpos.shape)
print("Super testing set shape:", super_testing_df3_QQpos.shape)
print("Any left in train?", any(train_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)))
print("Any left in test?", any(test_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)))

In [ ]:
# # Verification of the number of training and testing SMILES present
# print(train_data_df3_QQpos["SMILES_spectra"].nunique())
# print(test_data_df3_QQpos["SMILES_spectra"].nunique())
# # Limitation of the number of training and testing spectra used
# num_smiles_limit = 10  # Change this value to use a different number of SMILES
# unique_smiles = train_data_df3_QQpos['SMILES_spectra'].unique()[:num_smiles_limit]
# train_data_df3_QQpos = train_data_df3_QQpos[train_data_df3_QQpos['SMILES_spectra'].isin(unique_smiles)].copy()
# test_data_df3_QQpos = test_data_df3_QQpos[test_data_df3_QQpos['SMILES_spectra'].isin(unique_smiles)].copy()
# # Verify the limitation
# print(train_data_df3_QQpos["SMILES_spectra"].nunique())
# print(test_data_df3_QQpos["SMILES_spectra"].nunique())

In [ ]:
# Ensure we match the data correctly:
train_data = train_data_df3_QQpos
test_data = test_data_df3_QQpos
val_data = test_data
train_data_df3_QQpos_copy = train_data.copy()
test_data_df3_QQpos_copy = test_data.copy()


In [ ]:
train_data_df3_QQpos_copy.head()

In [ ]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_QQpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_QQpos = train_model_encoder(
    model=encoder_df3_QQpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)
x_val_df3_QQpos = x_val
x_train_df3_QQpos = x_train

### QQpos Testing

In [ ]:
# This is simply excecuting a test of the encoder, we do this process again down below when we want the encoder ouputs to be MLP inputs.
# Encoder excecution df3_QQpos
encoder_df3_QQpos.eval()
with torch.no_grad():
    test_output_df3_QQpos = encoder_df3_QQpos(x_val_df3_QQpos)  

with torch.no_grad():
    train_output_df3_QQpos = encoder_df3_QQpos(x_train_df3_QQpos)

# Encoder excecution df3_QQpos super testing
x_super_df3_QQpos = super_testing_df3_QQpos.iloc[:, 1:-1].values
x_super_tensor_df3_QQpos = torch.tensor(x_super_df3_QQpos, dtype=torch.float32).to(x_val_df3_QQpos.device)
encoder_df3_QQpos.eval()
with torch.no_grad():
    super_test_output_df3_QQpos = encoder_df3_QQpos(x_super_tensor_df3_QQpos)



### Training QTOFpos

In [ ]:
# Training and validation dataset split 
df3_QTOFpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QTOFpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QTOFpos_no_repeats.csv")
# Generalize the syntax
dataset =  df3_QTOFpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QTOFpos_copy = train_data.copy()
test_data_df3_QTOFpos_copy = test_data.copy()

# Save under correct names
train_data_df3_QTOFpos = train_data
test_data_df3_QTOFpos = test_data
# Load val_data
val_data = test_data

In [ ]:
# Remove the specified SMILES from the train and test sets,
# and collect them into a new DataFrame called super_testing_df3_QQpos

smiles_to_remove = ['CCCSP(=O)(OCC)SCCC', 'C#CCN(C)Cc1ccccc1', 'C1=CC(=CC=N1)C1=CC=NC=C1', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'C(NC1=NC=NC2=C1N=CN2C1CCCCO1)C1=CC=CC=C1', 'CC(C)N(CC1=CC=CC=C1)C(=O)C(C)(C)C', 'NC(=S)NC1=CC=CC=C1', 
                    'CC(C)(C)C1=CC(=CC(=C1O)C(C)(C)C)[N+]([O-])=O', 'CC(C)(C)C1=CC=C(C=C1)C(O)=O', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'CC(=C)C(=O)NC1=CC=C(Cl)C(Cl)=C1', 'CC(C)(C1=CC(Cl)=C(O)C(Cl)=C1)C1=CC(Cl)=C(O)C(Cl)=C1', 
                    'CCC(=O)N(C1CCN(CCC2=CC=CC=C2)CC1)C1=CC=CC=C1', 'CC(=O)N1CCN(CC1)C1=CC=C(OC[C@H]2CO[C@@](CN3C=CN=C3)(O2)C2=CC=C(Cl)C=C2Cl)C=C1', 
                    'CC(C)(C)C(=O)C(OC1=CC=C(Cl)C=C1)N1C=CN=C1', 'C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1', 'CC(=O)OCC[N+](C)(C)C', 
                    'CC(=O)NC1=CC=C(C=C1)S(=O)(=O)NC1=NOC(C)=C1', 'C1CCC(CC1)NC1CCCCC1', 'CC(CN1CCN(CCOCCO)CC1)CN1C2=CC=CC=C2SC2=CC=CC=C12', 
                    'CC(=O)OCC(COC(C)=O)OC(C)=O', 'CC(C)N(C(=O)CCl)c1ccccc1', 'CC1=C(C)N=C(NS(=O)(=O)C2=CC=C(N)C=C2)O1']

# Combine train and test to extract all rows with those SMILES
combined_df3_QTOFpos = pd.concat([train_data_df3_QTOFpos, test_data_df3_QTOFpos], ignore_index=True)
super_testing_df3_QTOFpos = combined_df3_QTOFpos[combined_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

# Remove those SMILES from train and test sets
train_data_df3_QTOFpos = train_data_df3_QTOFpos[~train_data_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)].copy()
test_data_df3_QTOFpos = test_data_df3_QTOFpos[~test_data_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

print("After removal, train shape:", train_data_df3_QTOFpos.shape)
print("After removal, test shape:", test_data_df3_QTOFpos.shape)
print("Super testing set shape:", super_testing_df3_QTOFpos.shape)
print("Any left in train?", any(train_data_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)))
print("Any left in test?", any(test_data_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)))

In [ ]:
# Ensure we match the data correctly:
train_data = train_data_df3_QTOFpos
test_data = test_data_df3_QTOFpos
val_data = test_data
train_data_df3_QTOFpos_copy = train_data.copy()
test_data_df3_QTOFpos_copy = test_data.copy()


In [ ]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_QTOFpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_QTOFpos = train_model_encoder(
    model=encoder_df3_QTOFpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)
x_val_df3_QTOFpos = x_val
x_train_df3_QTOFpos = x_train

### Encoder Training LTQO

In [ ]:
# Training and validation dataset split 
df3_LTQOpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_LTQOpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_LTQOpos_no_repeats.csv")
# Generalize the syntax
dataset =  df3_LTQOpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_LTQOpos_copy = train_data.copy()
test_data_df3_LTQOpos_copy = test_data.copy()

# Save under correct names
train_data_df3_LTQOpos = train_data
test_data_df3_LTQOpos = test_data
val_data = test_data

In [ ]:
# Remove the specified SMILES from the train and test sets,
# and collect them into a new DataFrame called super_testing_df3_QQpos

smiles_to_remove = ['CCCSP(=O)(OCC)SCCC', 'C#CCN(C)Cc1ccccc1', 'C1=CC(=CC=N1)C1=CC=NC=C1', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'C(NC1=NC=NC2=C1N=CN2C1CCCCO1)C1=CC=CC=C1', 'CC(C)N(CC1=CC=CC=C1)C(=O)C(C)(C)C', 'NC(=S)NC1=CC=CC=C1', 
                    'CC(C)(C)C1=CC(=CC(=C1O)C(C)(C)C)[N+]([O-])=O', 'CC(C)(C)C1=CC=C(C=C1)C(O)=O', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'CC(=C)C(=O)NC1=CC=C(Cl)C(Cl)=C1', 'CC(C)(C1=CC(Cl)=C(O)C(Cl)=C1)C1=CC(Cl)=C(O)C(Cl)=C1', 
                    'CCC(=O)N(C1CCN(CCC2=CC=CC=C2)CC1)C1=CC=CC=C1', 'CC(=O)N1CCN(CC1)C1=CC=C(OC[C@H]2CO[C@@](CN3C=CN=C3)(O2)C2=CC=C(Cl)C=C2Cl)C=C1', 
                    'CC(C)(C)C(=O)C(OC1=CC=C(Cl)C=C1)N1C=CN=C1', 'C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1', 'CC(=O)OCC[N+](C)(C)C', 
                    'CC(=O)NC1=CC=C(C=C1)S(=O)(=O)NC1=NOC(C)=C1', 'C1CCC(CC1)NC1CCCCC1', 'CC(CN1CCN(CCOCCO)CC1)CN1C2=CC=CC=C2SC2=CC=CC=C12', 
                    'CC(=O)OCC(COC(C)=O)OC(C)=O', 'CC(C)N(C(=O)CCl)c1ccccc1', 'CC1=C(C)N=C(NS(=O)(=O)C2=CC=C(N)C=C2)O1']

# Combine train and test to extract all rows with those SMILES
combined_df3_LTQOpos = pd.concat([train_data_df3_LTQOpos, test_data_df3_LTQOpos], ignore_index=True)
super_testing_df3_LTQOpos = combined_df3_LTQOpos[combined_df3_LTQOpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

# Remove those SMILES from train and test sets
train_data_df3_LTQOpos = train_data_df3_LTQOpos[~train_data_df3_LTQOpos['SMILES_spectra'].isin(smiles_to_remove)].copy()
test_data_df3_LTQOpos = test_data_df3_LTQOpos[~test_data_df3_LTQOpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

print("After removal, train shape:", train_data_df3_LTQOpos.shape)
print("After removal, test shape:", test_data_df3_LTQOpos.shape)
print("Super testing set shape:", super_testing_df3_LTQOpos.shape)
print("Any left in train?", any(train_data_df3_LTQOpos['SMILES_spectra'].isin(smiles_to_remove)))
print("Any left in test?", any(test_data_df3_QTOFpos['SMILES_spectra'].isin(smiles_to_remove)))

In [ ]:
# Ensure we match the data correctly:
train_data = train_data_df3_LTQOpos
test_data = test_data_df3_LTQOpos
val_data = test_data
train_data_df3_LTQOpos_copy = train_data.copy()
test_data_df3_LTQOpos_copy = test_data.copy()

In [ ]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_LTQOpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_LTQOpos = train_model_encoder(
    model=encoder_df3_LTQOpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)
x_val_df3_LTQOpos = x_val
x_train_df3_LTQOpos = x_train

# Spectra --> Toxicity MLP and Random Forest

### Processing

In [ ]:
# Add the 'Response' column using the SMILES as an identifier
def add_response_column_to_spectra(spectra_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds a 'Response' column to spectra_df by mapping from original_df using the SMILES column.
    """
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    spectra_df['Response'] = spectra_df[smiles_col].map(smiles_to_response)
    return spectra_df

df3_QQpos_spectra = add_response_and_log_response(df3_QQpos_spectra, df3_QQpos, smiles_col='SMILES_spectra') # Add response (needed to add EPA)
df3_QTOFpos_spectra = add_response_and_log_response(df3_QTOFpos_spectra, df3_QTOFpos, smiles_col='SMILES_spectra') # Add response (needed to add EPA)
df3_LTQOpos_spectra = add_response_and_log_response(df3_LTQOpos_spectra, df3_LTQOpos, smiles_col='SMILES_spectra') # Add response (needed to add EPA)

# Add EPA levels (one-hot encoded)
def add_epa_levels(df, response_col='Response', assign_func=assign_epa_level):
    """
    Adds EPA level columns (one-hot) to the DataFrame based on the response column.
    Removes the original response column.
    """
    df = df.copy()
    df["EPA_level"] = df[response_col].apply(assign_func)
    df = pd.get_dummies(df, columns=["EPA_level"], prefix='', prefix_sep='')
    epa_cols = [col for col in df.columns if str(col).startswith("EPA_level_")]
    df[epa_cols] = df[epa_cols].astype(int)
    #df.drop(columns=[response_col], inplace=True)
    return df

df3_QQpos_spectra_withEPA = add_epa_levels(df3_QQpos_spectra) # Add EPA, and then check to make sure
df3_QQpos_spectra_withEPA.head()

df3_QTOFpos_spectra_withEPA = add_epa_levels(df3_QTOFpos_spectra) # Add EPA, and then check to make sure
df3_LTQOpos_spectra_withEPA = add_epa_levels(df3_LTQOpos_spectra) # Add EPA, and then check to make sure


In [ ]:
# Super test removal, same as we had before with a even mix of EPA levels, though not all of these are in each group subset of the data.

smiles_to_remove = ['CCCSP(=O)(OCC)SCCC', 'C#CCN(C)Cc1ccccc1', 'C1=CC(=CC=N1)C1=CC=NC=C1', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'C(NC1=NC=NC2=C1N=CN2C1CCCCO1)C1=CC=CC=C1', 'CC(C)N(CC1=CC=CC=C1)C(=O)C(C)(C)C', 'NC(=S)NC1=CC=CC=C1', 
                    'CC(C)(C)C1=CC(=CC(=C1O)C(C)(C)C)[N+]([O-])=O', 'CC(C)(C)C1=CC=C(C=C1)C(O)=O', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'CC(=C)C(=O)NC1=CC=C(Cl)C(Cl)=C1', 'CC(C)(C1=CC(Cl)=C(O)C(Cl)=C1)C1=CC(Cl)=C(O)C(Cl)=C1', 
                    'CCC(=O)N(C1CCN(CCC2=CC=CC=C2)CC1)C1=CC=CC=C1', 'CC(=O)N1CCN(CC1)C1=CC=C(OC[C@H]2CO[C@@](CN3C=CN=C3)(O2)C2=CC=C(Cl)C=C2Cl)C=C1', 
                    'CC(C)(C)C(=O)C(OC1=CC=C(Cl)C=C1)N1C=CN=C1', 'C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1', 'CC(=O)OCC[N+](C)(C)C', 
                    'CC(=O)NC1=CC=C(C=C1)S(=O)(=O)NC1=NOC(C)=C1', 'C1CCC(CC1)NC1CCCCC1', 'CC(CN1CCN(CCOCCO)CC1)CN1C2=CC=CC=C2SC2=CC=CC=C12', 
                    'CC(=O)OCC(COC(C)=O)OC(C)=O', 'CC(C)N(C(=O)CCl)c1ccccc1', 'CC1=C(C)N=C(NS(=O)(=O)C2=CC=C(N)C=C2)O1']

# Combine train and test to extract all rows with those SMILES
super_testing_df3_QQpos = df3_QQpos_spectra_withEPA[df3_QQpos_spectra_withEPA['SMILES_spectra'].isin(smiles_to_remove)].copy()
super_testing_df3_QTOFpos = add_response_and_log_response(super_testing_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')
super_testing_df3_LTQOpos = add_response_and_log_response(super_testing_df3_LTQOpos, df3_LTQOpos, smiles_col='SMILES_spectra')

# Remove those SMILES from train and test sets
df3_QQpos_spectra_withEPA = df3_QQpos_spectra_withEPA[~df3_QQpos_spectra_withEPA['SMILES_spectra'].isin(smiles_to_remove)].copy()

print("After removal, shape:", df3_QQpos_spectra_withEPA.shape)
print("Super testing set shape:", super_testing_df3_QQpos.shape)
print("Any leff?", any(df3_QQpos_spectra_withEPA['SMILES_spectra'].isin(smiles_to_remove)))


### Random Forest Classifier

In [ ]:
# Count the number of samples in each EPA level in your dataset
epa_level_cols = [col for col in df3_QQpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]

# Sum each EPA level column to get counts
epa_counts = df3_QQpos_spectra_withEPA[epa_level_cols].sum().astype(int)
print("EPA level counts in test_output_df3_QQpos_withEPA:")
print(epa_counts)

# Identify embedding columns and EPA level columns
embedding_cols = [
    col for col in df3_QQpos_spectra_withEPA.columns
    if col not in ['SMILES'] + [col for col in df3_QQpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]
]
epa_level_cols = [col for col in df3_QQpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]
# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]


In [ ]:
# Balanced 4 way Random Forest Classifier
# Prepare features (X) and labels (y) as before
X = df3_QQpos_spectra_withEPA[embedding_cols].select_dtypes(include=[np.number])
X.columns = X.columns.astype(str)
y = df3_QQpos_spectra_withEPA[epa_level_cols].idxmax(axis=1)

# Set the target number of samples per class
target_count = 350
class_counts = dict(Counter(y))
sampling_strategy = {cls: min(target_count, class_counts[cls]) for cls in class_counts}
# First undersample to at most 30 per class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=15)
X_under, y_under = rus.fit_resample(X, y)

# Now oversample to exactly 30 per class
sampling_strategy = {cls: target_count for cls in class_counts}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=15)
X_balanced, y_balanced = ros.fit_resample(X_under, y_under)

# Train/test split
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=15, stratify=y_balanced
)

# Train Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=15)
rf_bal.fit(X_train_bal, y_train_bal)

# Predict and evaluate
y_pred_bal = rf_bal.predict(X_test_bal)
#print(confusion_matrix(y_test_bal, y_pred_bal))

In [ ]:
# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test_bal, y_pred_bal, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Balanced)")
plt.show()

### Random Forest Classifier QTOFpos

In [ ]:
# Count the number of samples in each EPA level in your dataset
epa_level_cols = [col for col in df3_QTOFpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]

# Sum each EPA level column to get counts
epa_counts = df3_QTOFpos_spectra_withEPA[epa_level_cols].sum().astype(int)
print("EPA level counts in test_output_df3_QTOFpos_withEPA:")
print(epa_counts)

# Identify embedding columns and EPA level columns
embedding_cols = [
    col for col in df3_QTOFpos_spectra_withEPA.columns
    if col not in ['SMILES'] + [col for col in df3_QTOFpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]
]
epa_level_cols = [col for col in df3_QTOFpos_spectra_withEPA.columns if str(col).startswith('EPA_level_')]
# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]

In [ ]:
# Balanced 4 way Random Forest Classifier
# Prepare features (X) and labels (y) as before
X = df3_QTOFpos_spectra_withEPA[embedding_cols].select_dtypes(include=[np.number])
X.columns = X.columns.astype(str)
y = df3_QTOFpos_spectra_withEPA[epa_level_cols].idxmax(axis=1)

# Set the target number of samples per class
target_count = 350
class_counts = dict(Counter(y))
sampling_strategy = {cls: min(target_count, class_counts[cls]) for cls in class_counts}
# First undersample to at most 30 per class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=5)
X_under, y_under = rus.fit_resample(X, y)

# Now oversample to exactly 30 per class
sampling_strategy = {cls: target_count for cls in class_counts}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=5)
X_balanced, y_balanced = ros.fit_resample(X_under, y_under)

# Train/test split
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=5, stratify=y_balanced
)

# Train Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=5)
rf_bal.fit(X_train_bal, y_train_bal)

# Predict and evaluate
y_pred_bal = rf_bal.predict(X_test_bal)
#print(confusion_matrix(y_test_bal, y_pred_bal))

In [ ]:
# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test_bal, y_pred_bal, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Balanced)")
plt.show()

### Spectra to Toxicity MLP Architecture

In [ ]:
batch_size = 128
epochs=1000
lr=0.0001
criterion=nn.MSELoss()
output_size = 1
num_layers = 10
#%%
# Everything below this line SHOULD be able to run without modification
class SpecToxMLP_Reg(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model_MLP_spectra(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_log_tox, _ in train_data:
            batch = batch.to(device)
            true_log_tox = true_log_tox.to(device)

            optimizer.zero_grad()
            batch_predicted_log_tox = model(batch)
            loss = criterion(batch_predicted_log_tox, true_log_tox)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_tox, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_tox = val_true_tox.to(device)

                val_batch_predicted_tox = model(val_batch)

                val_loss = criterion(val_batch_predicted_tox, val_true_tox)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model

### MLP Training QQpos 

In [ ]:
# Training and validation dataset split we need to add in response to the spectra.
df3_QQpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv")
df3_QQpos_spectra = add_response_and_log_response(df3_QQpos_spectra, df3_QQpos, smiles_col='SMILES_spectra')

# Generalize the syntax
dataset = df3_QQpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_data_df3_QQpos = train_data.copy()
test_data_df3_QQpos = test_data.copy()
val_data = test_data_df3_QQpos

# Make a copy
train_data_df3_QQpos_copy = train_data.copy()
test_data_df3_QQpos_copy = test_data.copy()

In [ ]:
# This isolates log response as the target varaible, not including embeddings at all.
def create_dataset_tensors_tox(spectra_dataset,device, start_idx=None, stop_idx=None):

    spectra = spectra_dataset.iloc[:,1:-3]

    # create tensors of spectra, true toxicity values, and chemical name encodings for train and val
    #chem_labels = list(spectra_dataset['SMILES_spectra'])
    log_tox_tensor = torch.Tensor(spectra_dataset["log_response"].values).unsqueeze(1).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return log_tox_tensor, spectra_tensor, spectra_indices_tensor

In [ ]:
# MLP training
device = f.set_up_gpu()
# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox(
    train_data, device, start_idx=2, stop_idx=-0)
del train_data

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox(
    val_data, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)


In [ ]:
#%%
SpecToxMLP_df3_QQpos = SpecToxMLP_Reg(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
SpecToxMLP_model_df3_QQpos = train_model_MLP_spectra(
    model=SpecToxMLP_df3_QQpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device)

df3_QQpos_x_val = x_val

### MLP Evaluation QQpos 

In [ ]:
# Spectra to tox MLP evaluation
SpecToxMLP_model_df3_QQpos.eval()
with torch.no_grad():
    test_output_df3_QQpos = SpecToxMLP_model_df3_QQpos(df3_QQpos_x_val)  


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Convert tensors to numpy arrays
y_true = y_val_tox.cpu().numpy().flatten()
y_pred = test_output_df3_QQpos.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
# Convert tensors to numpy arrays
y_true = y_val_tox.cpu().numpy().flatten()
y_pred = test_output_df3_QQpos.cpu().numpy().flatten()

# Remove outliers: keep only points within 3 standard deviations of the residuals
residuals = y_true - y_pred
std_res = np.std(residuals)
mask = np.abs(residuals) < 3 * std_res
y_true_filtered = y_true[mask]
y_pred_filtered = y_pred[mask]

# Scatter plot: predicted vs true (filtered)
plt.figure(figsize=(6,6))
plt.scatter(y_true_filtered, y_pred_filtered, alpha=0.5)
plt.plot([y_true_filtered.min(), y_true_filtered.max()], [y_true_filtered.min(), y_true_filtered.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted (Outliers Removed)')
plt.legend()
plt.show()

# Quantitative metrics (filtered)
mse = mean_squared_error(y_true_filtered, y_pred_filtered)
r2 = r2_score(y_true_filtered, y_pred_filtered)
print(f"Mean Squared Error (filtered): {mse:.4f}")
print(f"R^2 Score (filtered): {r2:.4f}")

### Spectra MLP on super test set QQpos 

In [ ]:
super_testing_df3_QQpos.head()

In [ ]:
# Ensure model and tensor are on the same device
model_device = next(SpecToxMLP_model_df3_QQpos.parameters()).device
add_response_and_log_response(super_testing_df3_QQpos, df3_QQpos, smiles_col='SMILES_spectra')

# Debug: Check what the model expects vs what we have
print(f"Super test dataset shape: {super_testing_df3_QQpos.shape}")
#print(f"Super test columns: {super_testing_df3_QQpos.columns.tolist()}")

# Get the expected input size from the model
model_input_size = next(SpecToxMLP_model_df3_QQpos.parameters()).shape[1]
print(f"Model expects input size: {model_input_size}")

# Use only numeric columns, excluding EPA levels and other non-spectral data
numeric_cols = super_testing_df3_QQpos.select_dtypes(include=[np.number]).columns.tolist()
# Remove EPA level columns and response columns
spectral_cols = [col for col in numeric_cols if not col.startswith('EPA_level_') 
                 and col not in ['Response', 'log_response']]

print(f"Available spectral columns: {len(spectral_cols)}")
print(f"First few spectral columns: {spectral_cols[:5]}")
print(f"Last few spectral columns: {spectral_cols[-5:]}")

# Take only the number of features the model expects
if len(spectral_cols) >= model_input_size:
    selected_cols = spectral_cols[:model_input_size]
    print(f"Using first {model_input_size} spectral columns")
else:
    selected_cols = spectral_cols
    print(f"Warning: Only {len(spectral_cols)} spectral columns available, but model expects {model_input_size}")

spectra_super = super_testing_df3_QQpos[selected_cols]
log_tox_val_super = super_testing_df3_QQpos['log_response']

print(f"Final spectra_super shape: {spectra_super.shape}")

# Convert the spectra to a tensor 
x_val_super_test = torch.Tensor(spectra_super.values).to(model_device)

# Now we can run the model on the super test set
SpecToxMLP_model_df3_QQpos.eval()
with torch.no_grad():
    super_test_output = SpecToxMLP_model_df3_QQpos(x_val_super_test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Convert tensors to numpy arrays
y_true = log_tox_val_super
y_pred = super_test_output.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg_QQpos = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg_QQpos.fit(X_train, y_train)


# Predict on super test set
y_pred_rf = rf_reg_QQpos.predict(spectra_super)
y_true = log_tox_val_super

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred_rf, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_true, y_pred_rf)
r2_rf = r2_score(y_true, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### Random forest regressor QQpos

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_reg.predict(X_test)

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### MLP Training QTOFpos

In [ ]:
# Training and validation dataset split we need to add in response to the spectra.
df3_QTOFpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QTOFpos_spectra.csv")
df3_QTOFpos_spectra = add_response_and_log_response(df3_QTOFpos_spectra, df3_QTOFpos, smiles_col='SMILES_spectra')

# Generalize the syntax
dataset = df3_QTOFpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_data_df3_QTOFpos = train_data.copy()
test_data_df3_QTOFpos = test_data.copy()
val_data = test_data_df3_QTOFpos

# Make a copy
train_data_df3_QTOFpos_copy = train_data.copy()
test_data_df3_QTOFpos_copy = test_data.copy()

In [ ]:
# MLP training
device = f.set_up_gpu()
# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox(
    train_data, device, start_idx=2, stop_idx=-0)
del train_data

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox(
    val_data, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

#%%
SpecToxMLP_df3_QTOFpos = SpecToxMLP_Reg(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
SpecToxMLP_model_df3_QTOFpos = train_model_MLP_spectra(
    model=SpecToxMLP_df3_QTOFpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device)

### MLP Evaluation QTOFpos

In [ ]:
# Spectra to tox MLP evaluation
SpecToxMLP_model_df3_QTOFpos.eval()
with torch.no_grad():
    test_output_df3_QTOFpos = SpecToxMLP_model_df3_QTOFpos(x_val)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Convert tensors to numpy arrays
y_true = y_val_tox.cpu().numpy().flatten()
y_pred = test_output_df3_QTOFpos.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
# Convert tensors to numpy arrays
y_true = y_val_tox.cpu().numpy().flatten()
y_pred = test_output_df3_QTOFpos.cpu().numpy().flatten()

# Remove outliers: keep only points within 3 standard deviations of the residuals
residuals = y_true - y_pred
std_res = np.std(residuals)
mask = np.abs(residuals) < 3 * std_res
y_true_filtered = y_true[mask]
y_pred_filtered = y_pred[mask]

# Scatter plot: predicted vs true (filtered)
plt.figure(figsize=(6,6))
plt.scatter(y_true_filtered, y_pred_filtered, alpha=0.5)
plt.plot([y_true_filtered.min(), y_true_filtered.max()], [y_true_filtered.min(), y_true_filtered.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted (Outliers Removed)')
plt.legend()
plt.show()

# Quantitative metrics (filtered)
mse = mean_squared_error(y_true_filtered, y_pred_filtered)
r2 = r2_score(y_true_filtered, y_pred_filtered)
print(f"Mean Squared Error (filtered): {mse:.4f}")
print(f"R^2 Score (filtered): {r2:.4f}")

### Spectra MLP on QTOF super test set

In [ ]:
# Ensure model and tensor are on the same device
model_device = next(SpecToxMLP_model_df3_QTOFpos.parameters()).device
add_response_and_log_response(super_testing_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')

# Debug: Check what the model expects vs what we have
print(f"Super test dataset shape: {super_testing_df3_QTOFpos.shape}")
#print(f"Super test columns: {super_testing_df3_QTOFpos.columns.tolist()}")

# Get the expected input size from the model
model_input_size = next(SpecToxMLP_model_df3_QTOFpos.parameters()).shape[1]
print(f"Model expects input size: {model_input_size}")

# Use only numeric columns, excluding EPA levels and other non-spectral data
numeric_cols = super_testing_df3_QTOFpos.select_dtypes(include=[np.number]).columns.tolist()
# Remove EPA level columns and response columns
spectral_cols = [col for col in numeric_cols if not col.startswith('EPA_level_') 
                 and col not in ['Response', 'log_response']]

print(f"Available spectral columns: {len(spectral_cols)}")
print(f"First few spectral columns: {spectral_cols[:5]}")
print(f"Last few spectral columns: {spectral_cols[-5:]}")

# Take only the number of features the model expects
if len(spectral_cols) >= model_input_size:
    selected_cols = spectral_cols[:model_input_size]
    print(f"Using first {model_input_size} spectral columns")
else:
    selected_cols = spectral_cols
    print(f"Warning: Only {len(spectral_cols)} spectral columns available, but model expects {model_input_size}")

spectra_super = super_testing_df3_QTOFpos[selected_cols]
log_tox_val_super = super_testing_df3_QTOFpos['log_response']

print(f"Final spectra_super shape: {spectra_super.shape}")

# Convert the spectra to a tensor 
x_val_super_test = torch.Tensor(spectra_super.values).to(model_device)

# Now we can run the model on the super test set
SpecToxMLP_model_df3_QTOFpos.eval()
with torch.no_grad():
    super_test_output = SpecToxMLP_model_df3_QTOFpos(x_val_super_test)

In [ ]:
# Convert tensors to numpy arrays
y_true = log_tox_val_super
y_pred = super_test_output.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)


# Predict on super test set
y_pred_rf = rf_reg.predict(spectra_super)
y_true = log_tox_val_super

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred_rf, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_true, y_pred_rf)
r2_rf = r2_score(y_true, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### Random Forest Regressor QTOFpos

In [ ]:
# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_reg.predict(X_test)

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### MLP Training LTQOpos

In [ ]:
# Training and validation dataset split we need to add in response to the spectra.
df3_LTQOpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_LTQOpos_spectra.csv")
df3_LTQOpos_spectra = add_response_and_log_response(df3_LTQOpos_spectra, df3_LTQOpos, smiles_col='SMILES_spectra')

# Generalize the syntax
dataset = df3_LTQOpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_data_df3_LTQOpos = train_data.copy()
test_data_df3_LTQOpos = test_data.copy()
val_data = test_data_df3_LTQOpos

# Make a copy
train_data_df3_LTQOpos_copy = train_data.copy()
test_data_df3_LTQOOpos_copy = test_data.copy()

In [ ]:
# MLP training
device = f.set_up_gpu()
# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox(
    train_data, device, start_idx=2, stop_idx=-0)
del train_data

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox(
    val_data, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

#%%
SpecToxMLP_df3_LTQOpos = SpecToxMLP_Reg(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
SpecToxMLP_model_df3_LTQOpos = train_model_MLP_spectra(
    model=SpecToxMLP_df3_LTQOpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device)

df3_LTQOpos_x_val = x_val
df3_LTQOpos_x_train = x_train

### MLP Evaluation LTQOpos

In [ ]:
# Spectra to tox MLP evaluation
SpecToxMLP_model_df3_LTQOpos.eval()
with torch.no_grad():
    test_output_df3_LTQOpos = SpecToxMLP_model_df3_LTQOpos(df3_LTQOpos_x_val)


In [ ]:
print(df3_LTQOpos_x_val.shape)
print(test_output_df3_LTQOpos.shape)
print(y_val_tox.shape) 
print(y_true.shape)
print(y_pred.shape)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Convert tensors to numpy arrays
y_true = y_val_tox.cpu().numpy().flatten()
y_pred = test_output_df3_LTQOpos.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

### Spectra MLP on LTQOpos super test set

In [ ]:
# CHECK FOR BUGS. DOESNT PROPERLY MATCH THE CHEMNET TWIN


# Ensure model and tensor are on the same device
model_device = next(SpecToxMLP_model_df3_LTQOpos.parameters()).device
add_response_and_log_response(super_testing_df3_LTQOpos, df3_LTQOpos, smiles_col='SMILES_spectra')

# Debug: Check what the model expects vs what we have
print(f"Super test dataset shape: {super_testing_df3_LTQOpos.shape}")
#print(f"Super test columns: {super_testing_df3_LTQOpos.columns.tolist()}")

# Get the expected input size from the model
model_input_size = next(SpecToxMLP_model_df3_LTQOpos.parameters()).shape[1]
print(f"Model expects input size: {model_input_size}")

# Use only numeric columns, excluding EPA levels and other non-spectral data
numeric_cols = super_testing_df3_LTQOpos.select_dtypes(include=[np.number]).columns.tolist()
# Remove EPA level columns and response columns
spectral_cols = [col for col in numeric_cols if not col.startswith('EPA_level_') 
                 and col not in ['Response', 'log_response']]

print(f"Available spectral columns: {len(spectral_cols)}")
print(f"First few spectral columns: {spectral_cols[:5]}")
print(f"Last few spectral columns: {spectral_cols[-5:]}")

# Take only the number of features the model expects
if len(spectral_cols) >= model_input_size:
    selected_cols = spectral_cols[:model_input_size]
    print(f"Using first {model_input_size} spectral columns")
else:
    selected_cols = spectral_cols
    print(f"Warning: Only {len(spectral_cols)} spectral columns available, but model expects {model_input_size}")

spectra_super = super_testing_df3_LTQOpos[selected_cols]
log_tox_val_super = super_testing_df3_LTQOpos['log_response']

print(f"Final spectra_super shape: {spectra_super.shape}")

# Convert the spectra to a tensor 
x_val_super_test = torch.Tensor(spectra_super.values).to(model_device)

# Now we can run the model on the super test set
SpecToxMLP_model_df3_LTQOpos.eval()
with torch.no_grad():
    super_test_output = SpecToxMLP_model_df3_LTQOpos(x_val_super_test)

In [ ]:
# Convert tensors to numpy arrays
y_true = log_tox_val_super
y_pred = super_test_output.cpu().numpy().flatten()

# Scatter plot: predicted vs true
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

In [ ]:
# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)


# Predict on super test set
y_pred_rf = rf_reg.predict(spectra_super)
y_true = log_tox_val_super

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred_rf, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_true, y_pred_rf)
r2_rf = r2_score(y_true, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### Random Forest Regressor LTQOpos

In [ ]:
# Convert tensors to numpy arrays for sklearn
X_train = x_train.cpu().numpy()
y_train = y_train_tox.cpu().numpy().flatten()
X_test = x_val.cpu().numpy()
y_test = y_val_tox.cpu().numpy().flatten()

# Train the Random Forest Regressor
rf_reg_LTQOpos = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg_LTQOpos.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_reg_LTQOpos.predict(X_test)

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Quantitative metrics
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

# ChemNet --> Toxicity MLP and Random Forest

### Processing to/of ChemNet Embeddings

In [ ]:
# Encoder excecution df3_QQpos
encoder_df3_QQpos.eval()
with torch.no_grad():
    test_output_df3_QQpos = encoder_df3_QQpos(x_val_df3_QQpos)  

with torch.no_grad():
    train_output_df3_QQpos = encoder_df3_QQpos(x_train_df3_QQpos)

# Encoder excecution df3_QQpos super testing
x_super_df3_QQpos = super_testing_df3_QQpos.iloc[:, 1:-6].values
x_super_tensor_df3_QQpos = torch.tensor(x_super_df3_QQpos, dtype=torch.float32).to(x_val_df3_QQpos.device)

encoder_df3_QQpos.eval()
with torch.no_grad():
    super_test_output_df3_QQpos = encoder_df3_QQpos(x_super_tensor_df3_QQpos)



In [ ]:
# Encoder excecution df3_QTOFpos
encoder_df3_QTOFpos.eval()
with torch.no_grad():
    test_output_df3_QTOFpos = encoder_df3_QTOFpos(x_val_df3_QTOFpos)

with torch.no_grad():
    train_output_df3_QTOFpos = encoder_df3_QTOFpos(x_train_df3_QTOFpos)

# Encoder excecution df3_QTOFpos super testing
x_super_df3_QTOFpos = super_testing_df3_QTOFpos.iloc[:, 1:-3].values
x_super_tensor_df3_QTOFpos = torch.tensor(x_super_df3_QTOFpos, dtype=torch.float32).to(x_val_df3_QTOFpos.device)
encoder_df3_QTOFpos.eval()
with torch.no_grad():
    super_test_output_df3_QTOFpos = encoder_df3_QTOFpos(x_super_tensor_df3_QTOFpos)


In [ ]:
# Encoder excecution df3_LTQOpos
encoder_df3_LTQOpos.eval()
with torch.no_grad():
    test_output_df3_LTQOpos = encoder_df3_LTQOpos(x_val_df3_LTQOpos)

with torch.no_grad():
    train_output_df3_LTQOpos = encoder_df3_LTQOpos(x_train_df3_LTQOpos)

# Encoder excecution df3_QTOFpos super testing
x_super_df3_LTQOpos = super_testing_df3_LTQOpos.iloc[:, 1:-3].values
x_super_tensor_df3_LTQOpos = torch.tensor(x_super_df3_LTQOpos, dtype=torch.float32).to(x_val_df3_LTQOpos.device)
encoder_df3_LTQOpos.eval()
with torch.no_grad():
    super_test_output_df3_LTQOpos = encoder_df3_LTQOpos(x_super_tensor_df3_LTQOpos)


In [ ]:
# Conversion of the outputs to numpy arrays then to pandas dataframes
test_output_df3_QQpos_np = test_output_df3_QQpos.cpu().numpy()
train_output_df3_QQpos_np = train_output_df3_QQpos.cpu().numpy()

# Convert to pandas DataFrames
test_output_df3_QQpos = pd.DataFrame(test_output_df3_QQpos_np, columns=[f'Embedding Float {i+1}' for i in range(test_output_df3_QQpos_np.shape[1])])
train_output_df3_QQpos = pd.DataFrame(train_output_df3_QQpos_np, columns=[f'Embedding Float {i+1}' for i in range(train_output_df3_QQpos_np.shape[1])])

# Ensure indices match before assignment
test_output_df3_QQpos = test_output_df3_QQpos.reset_index(drop=True)
test_data_df3_QQpos_copy = test_data_df3_QQpos_copy.reset_index(drop=True)
test_output_df3_QQpos['SMILES_spectra'] = test_data_df3_QQpos_copy['SMILES_spectra']

train_output_df3_QQpos = train_output_df3_QQpos.reset_index(drop=True)
train_data_df3_QQpos_copy = train_data_df3_QQpos_copy.reset_index(drop=True)
train_output_df3_QQpos['SMILES_spectra'] = train_data_df3_QQpos_copy['SMILES_spectra']


In [ ]:
# Conversion of the outputs to numpy arrays then to pandas dataframes
test_output_df3_QTOFpos_np = test_output_df3_QTOFpos.cpu().numpy()
train_output_df3_QTOFpos_np = train_output_df3_QTOFpos.cpu().numpy()

# Convert to pandas DataFrames
test_output_df3_QTOFpos = pd.DataFrame(test_output_df3_QTOFpos_np, columns=[f'Embedding Float {i+1}' for i in range(test_output_df3_QTOFpos_np.shape[1])])
train_output_df3_QTOFpos = pd.DataFrame(train_output_df3_QTOFpos_np, columns=[f'Embedding Float {i+1}' for i in range(train_output_df3_QTOFpos_np.shape[1])])

# Ensure indices match before assignment
test_output_df3_QTOFpos = test_output_df3_QTOFpos.reset_index(drop=True)
test_data_df3_QTOFpos_copy = test_data_df3_QTOFpos_copy.reset_index(drop=True)
test_output_df3_QTOFpos['SMILES_spectra'] = test_data_df3_QTOFpos_copy['SMILES_spectra']

train_output_df3_QTOFpos = train_output_df3_QTOFpos.reset_index(drop=True)
train_data_df3_QTOFpos_copy = train_data_df3_QTOFpos_copy.reset_index(drop=True)
train_output_df3_QTOFpos['SMILES_spectra'] = train_data_df3_QTOFpos_copy['SMILES_spectra']

In [ ]:
# Conversion of the outputs to numpy arrays then to pandas dataframes
test_output_df3_LTQOpos_np = test_output_df3_LTQOpos.cpu().numpy()
train_output_df3_LTQOpos_np = train_output_df3_LTQOpos.cpu().numpy()

# Convert to pandas DataFrames
test_output_df3_LTQOpos = pd.DataFrame(test_output_df3_LTQOpos_np, columns=[f'Embedding Float {i+1}' for i in range(test_output_df3_LTQOpos_np.shape[1])])
train_output_df3_LTQOpos = pd.DataFrame(train_output_df3_LTQOpos_np, columns=[f'Embedding Float {i+1}' for i in range(train_output_df3_LTQOpos_np.shape[1])])

# Ensure indices match before assignment
test_output_df3_LTQOpos = test_output_df3_LTQOpos.reset_index(drop=True)
test_data_df3_LTQOpos_copy = test_data_df3_LTQOpos_copy.reset_index(drop=True)
test_output_df3_LTQOpos['SMILES_spectra'] = test_data_df3_LTQOpos_copy['SMILES_spectra']

train_output_df3_LTQOpos = train_output_df3_LTQOpos.reset_index(drop=True)
train_data_df3_LTQOpos_copy = train_data_df3_LTQOpos_copy.reset_index(drop=True)
train_output_df3_LTQOpos['SMILES_spectra'] = train_data_df3_LTQOpos_copy['SMILES_spectra']

In [ ]:
# Add the 'Response' values to the test and train outputs
# def add_response_column(output_df, original_df, smiles_col='SMILES_spectra'):
#     """
#     Adds a 'Response' column to output_df by mapping from original_df using the SMILES column.
#     """
#     # Create mapping from SMILES to Response
#     smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
#     output_df['Response'] = output_df[smiles_col].map(smiles_to_response)
#     return output_df

# Example usage for each output DataFrame:
test_output_df3_QQpos = add_response_and_log_response(test_output_df3_QQpos, df3_QQpos, smiles_col='SMILES_spectra')
train_output_df3_QQpos = add_response_and_log_response(train_output_df3_QQpos, df3_QQpos, smiles_col='SMILES_spectra')

test_output_df3_QTOFpos = add_response_and_log_response(test_output_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')
train_output_df3_QTOFpos = add_response_and_log_response(train_output_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')

test_output_df3_LTQOpos = add_response_and_log_response(test_output_df3_LTQOpos, df3_LTQOpos, smiles_col='SMILES_spectra')
train_output_df3_LTQOpos = add_response_and_log_response(train_output_df3_LTQOpos, df3_LTQOpos, smiles_col='SMILES_spectra')


In [ ]:
def add_epa_levels(df, response_col='Response', assign_func=assign_epa_level):
    """
    Adds EPA level columns (one-hot) to the DataFrame based on the response column.
    Removes the original response column.
    """
    df = df.copy()
    # Assign EPA levels
    df["EPA_level"] = df[response_col].apply(assign_func)
    # One-hot encode
    df = pd.get_dummies(df, columns=["EPA_level"], prefix='', prefix_sep='')
    # Convert boolean columns to int
    epa_cols = [col for col in df.columns if str(col).startswith("EPA_level_")]
    df[epa_cols] = df[epa_cols].astype(int)
    # Remove the Response column
    #df.drop(columns=[response_col], inplace=True)
    return df

# Use it
test_output_df3_QQpos_withEPA = add_epa_levels(test_output_df3_QQpos)
train_output_df3_QQpos_withEPA = add_epa_levels(train_output_df3_QQpos)

test_output_df3_QTOFpos_withEPA = add_epa_levels(test_output_df3_QTOFpos)
train_output_df3_QTOFpos_withEPA = add_epa_levels(train_output_df3_QTOFpos)

test_output_df3_LTQOpos_withEPA = add_epa_levels(test_output_df3_LTQOpos)
train_output_df3_LTQOpos_withEPA = add_epa_levels(train_output_df3_LTQOpos)

In [ ]:
# Process encoder outputs for super test set
# QQpos
super_test_output_df3_QQpos_np = super_test_output_df3_QQpos.cpu().numpy()
super_test_output_df3_QQpos_df = pd.DataFrame(
    super_test_output_df3_QQpos_np, columns=[f'Embedding Float {i+1}' for i in range(super_test_output_df3_QQpos_np.shape[1])]
)
super_test_output_df3_QQpos_df['SMILES_spectra'] = super_testing_df3_QQpos['SMILES_spectra'].values

# Add the 'Response' column using the original dataframes
super_test_output_df3_QQpos_df = add_response_and_log_response(super_test_output_df3_QQpos_df, df3_QQpos, smiles_col='SMILES_spectra')

# Add EPA levels (one-hot) and remove 'Response'
super_test_output_df3_QQpos_withEPA = add_epa_levels(super_test_output_df3_QQpos_df)


In [ ]:
# QTOFpos
super_test_output_df3_QTOFpos_np = super_test_output_df3_QTOFpos.cpu().numpy()
super_test_output_df3_QTOFpos_df = pd.DataFrame(
    super_test_output_df3_QTOFpos_np, columns=[f'Embedding Float {i+1}' for i in range(super_test_output_df3_QTOFpos_np.shape[1])]
)
super_test_output_df3_QTOFpos_df['SMILES_spectra'] = super_testing_df3_QTOFpos['SMILES_spectra'].values

# Add the 'Response' column using the original dataframes
super_test_output_df3_QTOFpos_df = add_response_and_log_response(super_test_output_df3_QTOFpos_df, df3_QTOFpos, smiles_col='SMILES_spectra')

# Add EPA levels (one-hot) and remove 'Response'
super_test_output_df3_QTOFpos_withEPA = add_epa_levels(super_test_output_df3_QTOFpos_df)

In [ ]:
# LTQOpos
super_test_output_df3_LTQOpos_np = super_test_output_df3_LTQOpos.cpu().numpy()
super_test_output_df3_LTQOpos_df = pd.DataFrame(
    super_test_output_df3_LTQOpos_np, columns=[f'Embedding Float {i+1}' for i in range(super_test_output_df3_LTQOpos_np.shape[1])]
)
super_test_output_df3_LTQOpos_df['SMILES_spectra'] = super_testing_df3_LTQOpos['SMILES_spectra'].values

# Add the 'Response' column using the original dataframes
super_test_output_df3_LTQOpos_df = add_response_and_log_response(super_test_output_df3_LTQOpos_df, df3_LTQOpos, smiles_col='SMILES_spectra')

# Add EPA levels (one-hot) and remove 'Response'
super_test_output_df3_LTQOpos_withEPA = add_epa_levels(super_test_output_df3_LTQOpos_df)

### Random Forests

In [ ]:
# Count the number of samples in each EPA level in your dataset
epa_level_cols = [col for col in test_output_df3_QQpos_withEPA.columns if str(col).startswith('EPA_level_')]

# Sum each EPA level column to get counts
epa_counts = test_output_df3_QQpos_withEPA[epa_level_cols].sum().astype(int)
print("EPA level counts in test_output_df3_QQpos_withEPA:")
print(epa_counts)

In [ ]:
# Identify embedding columns and EPA level columns
embedding_cols = [
    col for col in test_output_df3_QQpos_withEPA.columns
    if col not in ['SMILES'] + [col for col in test_output_df3_QQpos_withEPA.columns if str(col).startswith('EPA_level_')]
]
epa_level_cols = [col for col in test_output_df3_QQpos_withEPA.columns if str(col).startswith('EPA_level_')]
# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]


In [ ]:
# Balanced 4 way Random Forest Classifier
# Prepare features (X) and labels (y) as before
X = test_output_df3_QQpos_withEPA[embedding_cols].select_dtypes(include=[np.number])
X.columns = X.columns.astype(str)
y = test_output_df3_QQpos_withEPA[epa_level_cols].idxmax(axis=1)

# Set the target number of samples per class
target_count = 350
class_counts = dict(Counter(y))
sampling_strategy = {cls: min(target_count, class_counts[cls]) for cls in class_counts}
# First undersample to at most 30 per class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=15)
X_under, y_under = rus.fit_resample(X, y)

# Now oversample to exactly 30 per class
sampling_strategy = {cls: target_count for cls in class_counts}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=15)
X_balanced, y_balanced = ros.fit_resample(X_under, y_under)

# Train/test split
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=15, stratify=y_balanced
)

# Train Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=15)
rf_bal.fit(X_train_bal, y_train_bal)

# Predict and evaluate
y_pred_bal = rf_bal.predict(X_test_bal)
#print(confusion_matrix(y_test_bal, y_pred_bal))

In [ ]:
# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test_bal, y_pred_bal, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Balanced)")
plt.show()

In [ ]:
# Count the number of samples in each EPA level in your dataset
epa_level_cols = [col for col in test_output_df3_QTOFpos_withEPA.columns if str(col).startswith('EPA_level_')]

# Sum each EPA level column to get counts
epa_counts = test_output_df3_QTOFpos_withEPA[epa_level_cols].sum().astype(int)
print("EPA level counts in test_output_df3_QTOFpos_withEPA:")
print(epa_counts)

In [ ]:
# Identify embedding columns and EPA level columns
embedding_cols = [
    col for col in test_output_df3_QTOFpos_withEPA.columns
    if col not in ['SMILES'] + [col for col in test_output_df3_QTOFpos_withEPA.columns if str(col).startswith('EPA_level_')]
]
epa_level_cols = [col for col in test_output_df3_QTOFpos_withEPA.columns if str(col).startswith('EPA_level_')]
# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]


In [ ]:
# Balanced 4 way Random Forest Classifier
# Prepare features (X) and labels (y) as before
X = test_output_df3_QTOFpos_withEPA[embedding_cols].select_dtypes(include=[np.number])
X.columns = X.columns.astype(str)
y = test_output_df3_QTOFpos_withEPA[epa_level_cols].idxmax(axis=1)

# Set the target number of samples per class
target_count = 350
class_counts = dict(Counter(y))
sampling_strategy = {cls: min(target_count, class_counts[cls]) for cls in class_counts}
# First undersample to at most 30 per class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=15)
X_under, y_under = rus.fit_resample(X, y)

# Now oversample to exactly 30 per class
sampling_strategy = {cls: target_count for cls in class_counts}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=15)
X_balanced, y_balanced = ros.fit_resample(X_under, y_under)

# Train/test split
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=15, stratify=y_balanced
)

# Train Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=15)
rf_bal.fit(X_train_bal, y_train_bal)

# Predict and evaluate
y_pred_bal = rf_bal.predict(X_test_bal)
#print(confusion_matrix(y_test_bal, y_pred_bal))

In [ ]:
# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test_bal, y_pred_bal, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Balanced)")
plt.show()

### Random Forest on super test set

In [ ]:
# Define spectra columns for super test set (exclude non-numeric and EPA columns)
spectra_cols = [
    col for col in super_test_output_df3_QQpos_withEPA.columns
    if col not in ['SMILES_spectra'] + [col for col in super_test_output_df3_QQpos_withEPA.columns if str(col).startswith('EPA_level_')]
]
X_super = super_test_output_df3_QQpos_withEPA[spectra_cols].select_dtypes(include=[np.number])
y_super = super_test_output_df3_QQpos_withEPA[epa_level_cols].idxmax(axis=1)

# Predict and plot confusion matrix
y_pred_super = rf_bal.predict(X_super)
cm_super = confusion_matrix(y_super, y_pred_super, labels=epa_order)
disp_super = ConfusionMatrixDisplay(confusion_matrix=cm_super, display_labels=epa_order)
disp_super.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Super Test Set)")
plt.show()

In [ ]:
# Now the same but with the random forest regressor
# Prepare X and y for train and test sets (ChemNet embeddings)
embedding_cols = [col for col in train_output_df3_QQpos.columns if col.startswith('Embedding Float')]
X_train_rf = train_output_df3_QQpos[embedding_cols].values
y_train_rf = train_output_df3_QQpos['log_response'].values

X_test_rf = test_output_df3_QQpos[embedding_cols].values
y_test_rf = test_output_df3_QQpos['log_response'].values

# Combine train and test for training
X_rf = np.vstack([X_train_rf, X_test_rf])
y_rf = np.concatenate([y_train_rf, y_test_rf])

# Prepare super test set
X_super_rf = super_test_output_df3_QQpos_withEPA[embedding_cols].values
y_super_rf = super_test_output_df3_QQpos_withEPA['log_response'].values

# Train the Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_rf, y_rf)

# Predict on super test set
y_pred_super_rf = rf_reg.predict(X_super_rf)

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_super_rf, y_pred_super_rf, alpha=0.5)
plt.plot([y_super_rf.min(), y_super_rf.max()], [y_super_rf.min(), y_super_rf.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: Super Test Set')
plt.legend()
plt.show()

# Quantitative metrics
mse_super = mean_squared_error(y_super_rf, y_pred_super_rf)
r2_super = r2_score(y_super_rf, y_pred_super_rf)
print(f"Super Test Set Mean Squared Error: {mse_super:.4f}")
print(f"Super Test Set R^2 Score: {r2_super:.4f}")

In [ ]:
# Define spectra columns for super test set (exclude non-numeric and EPA columns)
spectra_cols = [
    col for col in super_test_output_df3_QTOFpos_withEPA.columns
    if col not in ['SMILES_spectra'] + [col for col in super_test_output_df3_QTOFpos_withEPA.columns if str(col).startswith('EPA_level_')]
]
X_super = super_test_output_df3_QTOFpos_withEPA[spectra_cols].select_dtypes(include=[np.number])
y_super = super_test_output_df3_QTOFpos_withEPA[epa_level_cols].idxmax(axis=1)

# Predict and plot confusion matrix
y_pred_super = rf_bal.predict(X_super)
cm_super = confusion_matrix(y_super, y_pred_super, labels=epa_order)
disp_super = ConfusionMatrixDisplay(confusion_matrix=cm_super, display_labels=epa_order)
disp_super.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Super Test Set)")
plt.show()

In [ ]:
# Define spectra columns for super test set (exclude non-numeric and EPA columns)
spectra_cols = [
    col for col in super_test_output_df3_QQpos_withEPA.columns
    if col not in ['SMILES_spectra'] + [col for col in super_test_output_df3_QQpos_withEPA.columns if str(col).startswith('EPA_level_')]
]
X_super = super_test_output_df3_QQpos_withEPA[spectra_cols].select_dtypes(include=[np.number])
y_super = super_test_output_df3_QQpos_withEPA[epa_level_cols].idxmax(axis=1)

# Predict and plot confusion matrix
y_pred_super = rf_bal.predict(X_super)
cm_super = confusion_matrix(y_super, y_pred_super, labels=epa_order)
disp_super = ConfusionMatrixDisplay(confusion_matrix=cm_super, display_labels=epa_order)
disp_super.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Super Test Set)")
plt.show()

### MLP Data Processing

In [ ]:
# Encoder excecution 
encoder_df3_QQpos.eval()
with torch.no_grad():
    test_output_df3_QQpos = encoder_df3_QQpos(x_val_df3_QQpos)  

with torch.no_grad():
    train_output_df3_QQpos = encoder_df3_QQpos(x_train_df3_QQpos)

# Conversion of the outputs to numpy arrays then to pandas dataframes
test_output_df3_QQpos_np = test_output_df3_QQpos.cpu().numpy()
train_output_df3_QQpos_np = train_output_df3_QQpos.cpu().numpy()

# Convert to pandas DataFrames
test_output_df3_QQpos = pd.DataFrame(test_output_df3_QQpos_np, columns=[f'Embedding Float {i+1}' for i in range(test_output_df3_QQpos_np.shape[1])])
train_output_df3_QQpos = pd.DataFrame(train_output_df3_QQpos_np, columns=[f'Embedding Float {i+1}' for i in range(train_output_df3_QQpos_np.shape[1])])

# Add the 'SMILES_spectra' column to the test and train outputs using the indices from the original test dataframes
test_output_df3_QQpos['SMILES_spectra'] = test_data_df3_QQpos_copy['SMILES_spectra']
train_output_df3_QQpos['SMILES_spectra'] = train_data_df3_QQpos_copy['SMILES_spectra']

# Add the 'Response' values to the test and train outputs
def add_response_column(output_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds a 'Response' column to output_df by mapping from original_df using the SMILES column.
    """
    # Create mapping from SMILES to Response
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    output_df['Response'] = output_df[smiles_col].map(smiles_to_response)
    return output_df

# Add the response column
test_output_df3_QQpos = add_response_and_log_response(test_output_df3_QQpos, df3_QQpos, smiles_col='SMILES_spectra')
train_output_df3_QQpos = add_response_and_log_response(train_output_df3_QQpos, df3_QQpos, smiles_col='SMILES_spectra')


# Add an index for bette integration into the MLP algorithm
test_output_df3_QQpos['index'] = test_output_df3_QQpos.index
train_output_df3_QQpos['index'] = train_output_df3_QQpos.index

# Save a copy
test_output_df3_QQpos_copy = test_output_df3_QQpos.copy()
train_output_df3_QQpos_copy = train_output_df3_QQpos.copy()

In [ ]:
# Encoder excecution 
encoder_df3_QTOFpos.eval()
with torch.no_grad():
    test_output_df3_QTOFpos = encoder_df3_QTOFpos(x_val_df3_QTOFpos)  

with torch.no_grad():
    train_output_df3_QTOFpos = encoder_df3_QTOFpos(x_train_df3_QTOFpos)

# Conversion of the outputs to numpy arrays then to pandas dataframes
test_output_df3_QTOFpos_np = test_output_df3_QTOFpos.cpu().numpy()
train_output_df3_QTOFpos_np = train_output_df3_QTOFpos.cpu().numpy()

# Convert to pandas DataFrames
test_output_df3_QTOFpos = pd.DataFrame(test_output_df3_QTOFpos_np, columns=[f'Embedding Float {i+1}' for i in range(test_output_df3_QTOFpos_np.shape[1])])
train_output_df3_QTOFpos = pd.DataFrame(train_output_df3_QTOFpos_np, columns=[f'Embedding Float {i+1}' for i in range(train_output_df3_QTOFpos_np.shape[1])])

# Add the 'SMILES_spectra' column to the test and train outputs using the indices from the original test dataframes
test_output_df3_QTOFpos['SMILES_spectra'] = test_data_df3_QTOFpos_copy['SMILES_spectra']
train_output_df3_QTOFpos['SMILES_spectra'] = train_data_df3_QTOFpos_copy['SMILES_spectra']

# Add the 'Response' values to the test and train outputs
def add_response_column(output_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds a 'Response' column to output_df by mapping from original_df using the SMILES column.
    """
    # Create mapping from SMILES to Response
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    output_df['Response'] = output_df[smiles_col].map(smiles_to_response)
    return output_df

# Add the response column
test_output_df3_QTOFpos = add_response_and_log_response(test_output_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')
train_output_df3_QTOFpos = add_response_and_log_response(train_output_df3_QTOFpos, df3_QTOFpos, smiles_col='SMILES_spectra')


# Add an index for better integration into the MLP algorithm
test_output_df3_QTOFpos['index'] = test_output_df3_QTOFpos.index
train_output_df3_QTOFpos['index'] = train_output_df3_QTOFpos.index

# Save a copy
test_output_df3_QTOFpos_copy = test_output_df3_QTOFpos.copy()
train_output_df3_QTOFpos_copy = train_output_df3_QTOFpos.copy()

In [ ]:
print(test_output_df3_QQpos.shape)
test_output_df3_QQpos.head()

In [ ]:
print(train_output_df3_QQpos.shape)
train_output_df3_QQpos.head()

### MLP Architecture

In [ ]:
# The genreal architecture of the MLP will be the same as that of the encoder, the main difference between the two is that the MLP will 
# have a 1 dimensional output, the true Response values of each SMILES, and we will use those SMILES to track which repsonse value belongs 
# with each Chemical.
# We will also adjust the input so that we have the log(response) rather than just response as our varaible of interest.

#%%
epochs=100
lr=0.0001
criterion=nn.MSELoss()
output_size = 1
num_layers = 5
#%%

# Everything below this line SHOULD be able to run without modification
class ToxMLP(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model_MLP(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_log_tox, _ in train_data:
            batch = batch.to(device)
            true_log_tox = true_log_tox.to(device)

            optimizer.zero_grad()
            batch_predicted_log_tox = model(batch)
            loss = criterion(batch_predicted_log_tox, true_log_tox)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_tox, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_tox = val_true_tox.to(device)

                val_batch_predicted_tox = model(val_batch)

                val_loss = criterion(val_batch_predicted_tox, val_true_tox)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model

### MLP Data Prep

In [ ]:
def create_dataset_tensors_tox(spectra_dataset,device, start_idx=None, stop_idx=None):

    spectra = spectra_dataset.iloc[:,1:-4]

    # create tensors of spectra, true toxicity values, and chemical name encodings for train and val
    #chem_labels = list(spectra_dataset['SMILES_spectra'])
    log_tox_tensor = torch.Tensor(spectra_dataset["log_response"].values).unsqueeze(1).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return log_tox_tensor, spectra_tensor, spectra_indices_tensor

def create_dataset_tensors_tox_spec(spectra_dataset,device, start_idx=None, stop_idx=None):

    embedding_cols = [col for col in spectra_dataset.columns if col.startswith('Embedding Float')]
    spectra = spectra_dataset[embedding_cols]

    # create tensors of spectra, true toxicity values, and chemical name encodings for train and val
    #chem_labels = list(spectra_dataset['SMILES_spectra'])
    log_tox_tensor = torch.Tensor(spectra_dataset["log_response"].values).unsqueeze(1).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return log_tox_tensor, spectra_tensor, spectra_indices_tensor

### MLP Training QQpos

In the training process we should combine the set, resplit them as we have done in the past then we can train and evaulate the model. 

In [ ]:
combined_output_df3_QQpos = pd.concat([train_output_df3_QQpos, test_output_df3_QQpos], ignore_index=True)
print(combined_output_df3_QQpos.shape)
combined_output_df3_QQpos.head()

In [ ]:
# Generalize the syntax
dataset = combined_output_df3_QQpos

train_indices = []
test_indices = []

for smiles, group in dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = dataset.loc[train_indices].reset_index(drop=True)
test_data = dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
#train_data['index'] = train_data.index
#test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_output_df3_QQpos = train_data.copy()
test_output_df3_QQpos = test_data.copy()
val_data = test_output_df3_QQpos

# Make a copy
train_output_df3_QQpos_copy = train_data.copy()
test_output_df3_QQpos_copy = test_data.copy()

In [ ]:
print(train_output_df3_QQpos.shape)
train_output_df3_QQpos.head()

In [ ]:
# MLP training
device = f.set_up_gpu()

# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox_spec(
    train_output_df3_QQpos, device, start_idx=2, stop_idx=-0)
del train_output_df3_QQpos

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox_spec(
    test_output_df3_QQpos, device, start_idx=2, stop_idx=-0)
del test_output_df3_QQpos


In [ ]:
print(x_train.shape, y_train_tox.shape, train_indices_tensor.shape)

In [ ]:

train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
toxMLP_df3_QQpos = ToxMLP(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
MLP_model_df3_QQpos = train_model_MLP(
    model=toxMLP_df3_QQpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

### MLP Evaluation QQpos

In [ ]:
# Excecution
MLP_model_df3_QQpos.eval()
with torch.no_grad():
    ChemNet_MLP_test_output_df3_QQpos = toxMLP_df3_QQpos(x_val) 

# Excecution
MLP_model_df3_QQpos.eval()
with torch.no_grad():
    ChemNet_MLP_train_output_df3_QQpos = toxMLP_df3_QQpos(x_train)  
 

In [ ]:
print(x_val.shape)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Use the correct source for true values
y_true_test = test_output_df3_QQpos_copy['log_response'].values
y_pred_test = ChemNet_MLP_test_output_df3_QQpos.cpu().numpy().flatten()

plt.figure(figsize=(6,6))
plt.scatter(y_true_test, y_pred_test, alpha=0.5)
plt.plot([y_true_test.min(), y_true_test.max()], [y_true_test.min(), y_true_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

mse_test = mean_squared_error(y_true_test, y_pred_test)
r2_test = r2_score(y_true_test, y_pred_test)
print(f"Test Set Mean Squared Error: {mse_test:.4f}")
print(f"Test Set R^2 Score: {r2_test:.4f}")


In [ ]:
# Use the correct source for true values
y_true_train = train_output_df3_QQpos_copy['log_response'].values
y_pred_train = ChemNet_MLP_train_output_df3_QQpos.cpu().numpy().flatten()

plt.figure(figsize=(6,6))
plt.scatter(y_true_train, y_pred_train, alpha=0.5)
plt.plot([y_true_train.min(), y_true_train.max()], [y_true_train.min(), y_true_train.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted (Training)')
plt.legend()
plt.show()

mse_train = mean_squared_error(y_true_train, y_pred_train)
r2_train = r2_score(y_true_train, y_pred_train)
print(f"Train Set Mean Squared Error: {mse_train:.4f}")
print(f"Train Set R^2 Score: {r2_train:.4f}")

### MLP on the super test set QQpos

In [ ]:
print(super_test_output_df3_QQpos_withEPA.shape)
super_test_output_df3_QQpos_withEPA.head()

In [ ]:
# --- ChemNet MLP: Predict and evaluate on super test set (NO retraining) ---

# Prepare X for the super test set (ChemNet embeddings)
embedding_cols = [col for col in super_test_output_df3_QQpos_withEPA.columns if col.startswith('Embedding Float')]
X_super = super_test_output_df3_QQpos_withEPA[embedding_cols].values
y_super = super_test_output_df3_QQpos_withEPA['log_response'].values

# Debug information
print(f"Training data shape: {x_train.shape}")
print(f"Super test set shape: {X_super.shape}")
print(f"Number of embedding columns in super test set: {len(embedding_cols)}")
print(f"MLP model expects input size: {x_train.shape[1]}")
print(f"Super test set has: {X_super.shape[1]} features")

# Check if we need to adjust the feature size
if X_super.shape[1] != x_train.shape[1]:
    print(f"Shape mismatch! Need to adjust from {X_super.shape[1]} to {x_train.shape[1]} features")
    
    # Option 1: Truncate the super test set to match training data size
    if X_super.shape[1] > x_train.shape[1]:
        print("Truncating super test set to match training data size")
        X_super = X_super[:, :x_train.shape[1]]
    
    # Option 2: Pad the super test set if it's smaller (unlikely in this case)
    elif X_super.shape[1] < x_train.shape[1]:
        print("Padding super test set to match training data size")
        padding = np.zeros((X_super.shape[0], x_train.shape[1] - X_super.shape[1]))
        X_super = np.concatenate([X_super, padding], axis=1)

print(f"Final super test set shape: {X_super.shape}")

# Convert to tensor and move to device
x_super_tensor = torch.tensor(X_super, dtype=torch.float32).to(device)

# Predict with the already-trained MLP
MLP_model_df3_QQpos.eval()
with torch.no_grad():
    y_pred_super = MLP_model_df3_QQpos(x_super_tensor).cpu().numpy().flatten()

# Visualization and metrics
plt.figure(figsize=(6,6))
plt.scatter(y_super, y_pred_super, alpha=0.5)
plt.plot([y_super.min(), y_super.max()], [y_super.min(), y_super.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('MLP Predicted log(Response)')
plt.title('MLP Regression: Super Test Set')
plt.legend()
plt.show()

mse_super = mean_squared_error(y_super, y_pred_super)
r2_super = r2_score(y_super, y_pred_super)
print(f"Super Test Set Mean Squared Error: {mse_super:.4f}")
print(f"Super Test Set R^2 Score: {r2_super:.4f}")

### Random forest regressor QQpos

In [ ]:
# Prepare X and y for the Random Forest
X_rf = test_output_df3_QQpos_copy.iloc[:, :-4].select_dtypes(include=[float, int]).values
y_rf = test_output_df3_QQpos_copy['log_response'].values

# Train/test split (here, use all for both train and test for direct comparison, or split as needed)
# For demonstration, train and test on the same set
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_rf, y_rf)
y_pred_rf = rf_reg.predict(X_rf)

# Plotting
plt.figure(figsize=(6,6))
plt.scatter(y_rf, y_pred_rf, alpha=0.5)
plt.plot([y_rf.min(), y_rf.max()], [y_rf.min(), y_rf.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Metrics
mse_rf = mean_squared_error(y_rf, y_pred_rf)
r2_rf = r2_score(y_rf, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### MLP Training QTOFpos

In [ ]:
combined_output_df3_QTOFpos = pd.concat([train_output_df3_QTOFpos, test_output_df3_QTOFpos], ignore_index=True)
print(combined_output_df3_QTOFpos.shape)
combined_output_df3_QTOFpos.head()

In [ ]:
# Generalize the syntax
dataset = combined_output_df3_QTOFpos

train_indices = []
test_indices = []

for smiles, group in dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = dataset.loc[train_indices].reset_index(drop=True)
test_data = dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
#train_data['index'] = train_data.index
#test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_output_df3_QTOFpos = train_data.copy()
test_output_df3_QTOFpos = test_data.copy()
val_data = test_output_df3_QTOFpos

# Make a copy
train_output_df3_QTOFpos_copy = train_data.copy()
test_output_df3_QTOFpos_copy = test_data.copy()

In [ ]:
# MLP training
device = f.set_up_gpu()

# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox_spec(
    train_output_df3_QTOFpos, device, start_idx=2, stop_idx=-0)
del train_output_df3_QTOFpos

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox_spec(
    test_output_df3_QTOFpos, device, start_idx=2, stop_idx=-0)
del test_output_df3_QTOFpos


In [ ]:
train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
toxMLP_df3_QTOFpos = ToxMLP(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
MLP_model_df3_QTOFpos = train_model_MLP(
    model=toxMLP_df3_QTOFpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

### MLP Evaluation QTOFpos

In [ ]:
# Excecution
MLP_model_df3_QTOFpos.eval()
with torch.no_grad():
    ChemNet_MLP_test_output_df3_QTOFpos = toxMLP_df3_QTOFpos(x_val) 

# Excecution
MLP_model_df3_QTOFpos.eval()
with torch.no_grad():
    ChemNet_MLP_train_output_df3_QTOFpos = toxMLP_df3_QTOFpos(x_train)


In [ ]:
# Use the correct source for true values
y_true_test = test_output_df3_QTOFpos_copy['log_response'].values
y_pred_test = ChemNet_MLP_test_output_df3_QTOFpos.cpu().numpy().flatten()

# DEBUG: Check sizes
print(f"y_true_test shape: {y_true_test.shape}")
print(f"y_pred_test shape: {y_pred_test.shape}")
print(f"test_output_df3_QTOFpos_copy shape: {test_output_df3_QTOFpos_copy.shape}")

# FIX: Align the arrays to the same size
min_length = min(len(y_true_test), len(y_pred_test))
y_true_test = y_true_test[:min_length]
y_pred_test = y_pred_test[:min_length]

print(f"After alignment: y_true={len(y_true_test)}, y_pred={len(y_pred_test)}")

plt.figure(figsize=(6,6))
plt.scatter(y_true_test, y_pred_test, alpha=0.5)
plt.plot([y_true_test.min(), y_true_test.max()], [y_true_test.min(), y_true_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

mse_test = mean_squared_error(y_true_test, y_pred_test)
r2_test = r2_score(y_true_test, y_pred_test)
print(f"Test Set Mean Squared Error: {mse_test:.4f}")
print(f"Test Set R^2 Score: {r2_test:.4f}")


In [ ]:
# Use the correct source for true values
y_true_train = train_output_df3_QTOFpos_copy['log_response'].values
y_pred_train = ChemNet_MLP_train_output_df3_QTOFpos.cpu().numpy().flatten()

plt.figure(figsize=(6,6))
plt.scatter(y_true_train, y_pred_train, alpha=0.5)
plt.plot([y_true_train.min(), y_true_train.max()], [y_true_train.min(), y_true_train.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted (Training)')
plt.legend()
plt.show()

mse_train = mean_squared_error(y_true_train, y_pred_train)
r2_train = r2_score(y_true_train, y_pred_train)
print(f"Train Set Mean Squared Error: {mse_train:.4f}")
print(f"Train Set R^2 Score: {r2_train:.4f}")

### MLP on super test set QTOFpos

In [ ]:
# --- ChemNet MLP: Predict and evaluate on super test set (NO retraining) ---

# Prepare X for the super test set (ChemNet embeddings)
embedding_cols = [col for col in super_test_output_df3_QTOFpos_withEPA.columns if col.startswith('Embedding Float')]
X_super = super_test_output_df3_QTOFpos_withEPA[embedding_cols].values
y_super = super_test_output_df3_QTOFpos_withEPA['log_response'].values

# Debug information
print(f"Training data shape: {x_train.shape}")
print(f"Super test set shape: {X_super.shape}")
print(f"Number of embedding columns in super test set: {len(embedding_cols)}")
print(f"MLP model expects input size: {x_train.shape[1]}")
print(f"Super test set has: {X_super.shape[1]} features")

# Check if we need to adjust the feature size
if X_super.shape[1] != x_train.shape[1]:
    print(f"Shape mismatch! Need to adjust from {X_super.shape[1]} to {x_train.shape[1]} features")
    
    # Option 1: Truncate the super test set to match training data size
    if X_super.shape[1] > x_train.shape[1]:
        print("Truncating super test set to match training data size")
        X_super = X_super[:, :x_train.shape[1]]
    
    # Option 2: Pad the super test set if it's smaller (unlikely in this case)
    elif X_super.shape[1] < x_train.shape[1]:
        print("Padding super test set to match training data size")
        padding = np.zeros((X_super.shape[0], x_train.shape[1] - X_super.shape[1]))
        X_super = np.concatenate([X_super, padding], axis=1)

print(f"Final super test set shape: {X_super.shape}")

# Convert to tensor and move to device
x_super_tensor = torch.tensor(X_super, dtype=torch.float32).to(device)

# Predict with the already-trained MLP
MLP_model_df3_QTOFpos.eval()
with torch.no_grad():
    y_pred_super = MLP_model_df3_QTOFpos(x_super_tensor).cpu().numpy().flatten()

# Visualization and metrics
plt.figure(figsize=(6,6))
plt.scatter(y_super, y_pred_super, alpha=0.5)
plt.plot([y_super.min(), y_super.max()], [y_super.min(), y_super.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('MLP Predicted log(Response)')
plt.title('MLP Regression: Super Test Set')
plt.legend()
plt.show()

mse_super = mean_squared_error(y_super, y_pred_super)
r2_super = r2_score(y_super, y_pred_super)
print(f"Super Test Set Mean Squared Error: {mse_super:.4f}")
print(f"Super Test Set R^2 Score: {r2_super:.4f}")

### Random forest regressor QTOFpos

In [ ]:
# Prepare X and y for the Random Forest
X_rf = test_output_df3_QTOFpos_copy.iloc[:, :-4].select_dtypes(include=[float, int]).values
y_rf = test_output_df3_QTOFpos_copy['log_response'].values

# Train/test split (here, use all for both train and test for direct comparison, or split as needed)
# For demonstration, train and test on the same set
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_rf, y_rf)
y_pred_rf = rf_reg.predict(X_rf)

# Plotting
plt.figure(figsize=(6,6))
plt.scatter(y_rf, y_pred_rf, alpha=0.5)
plt.plot([y_rf.min(), y_rf.max()], [y_rf.min(), y_rf.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Metrics
mse_rf = mean_squared_error(y_rf, y_pred_rf)
r2_rf = r2_score(y_rf, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

### MLP Training LTQOpos

In [ ]:
combined_output_df3_LTQOpos = pd.concat([train_output_df3_LTQOpos, test_output_df3_LTQOpos], ignore_index=True)
print(combined_output_df3_LTQOpos.shape)
combined_output_df3_LTQOpos.head()

In [ ]:
# Generalize the syntax
dataset = combined_output_df3_LTQOpos

train_indices = []
test_indices = []

for smiles, group in dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = dataset.loc[train_indices].reset_index(drop=True)
test_data = dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

# Save the dataframes with more descriptive names
train_output_df3_LTQOpos = train_data.copy()
test_output_df3_LTQOpos = test_data.copy()
val_data = test_output_df3_LTQOpos

# Make a copy
train_output_df3_LTQOpos_copy = train_data.copy()
test_output_df3_LTQOpos_copy = test_data.copy()

In [ ]:
# MLP training
device = f.set_up_gpu()

# Training set
y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_tox_spec(
    train_output_df3_LTQOpos, device, start_idx=2, stop_idx=-0)
del train_output_df3_LTQOpos

# Validation set
y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_tox_spec(
    test_output_df3_LTQOpos, device, start_idx=2, stop_idx=-0)
del test_output_df3_LTQOpos


In [ ]:
train_data = TensorDataset(x_train, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
toxMLP_df3_LTQOpos = ToxMLP(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
MLP_model_df3_LTQOpos = train_model_MLP(
    model=toxMLP_df3_LTQOpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)


### MLP Evaluation LTQOpos

In [ ]:
# Excecution
MLP_model_df3_LTQOpos.eval()
with torch.no_grad():
    ChemNet_MLP_test_output_df3_LTQOpos = toxMLP_df3_LTQOpos(x_val) 

# Excecution
MLP_model_df3_LTQOpos.eval()
with torch.no_grad():
    ChemNet_MLP_train_output_df3_LTQOpos = toxMLP_df3_LTQOpos(x_train)


In [ ]:
# Use the correct source for true values
y_true_test = test_output_df3_LTQOpos_copy['log_response'].values
y_pred_test = ChemNet_MLP_test_output_df3_LTQOpos.cpu().numpy().flatten()

# DEBUG: Check sizes
print(f"y_true_test shape: {y_true_test.shape}")
print(f"y_pred_test shape: {y_pred_test.shape}")
print(f"test_output_df3_LTQOpos_copy shape: {test_output_df3_LTQOpos_copy.shape}")

# FIX: Align the arrays to the same size
min_length = min(len(y_true_test), len(y_pred_test))
y_true_test = y_true_test[:min_length]
y_pred_test = y_pred_test[:min_length]

print(f"After alignment: y_true={len(y_true_test)}, y_pred={len(y_pred_test)}")

plt.figure(figsize=(6,6))
plt.scatter(y_true_test, y_pred_test, alpha=0.5)
plt.plot([y_true_test.min(), y_true_test.max()], [y_true_test.min(), y_true_test.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('MLP Regression: True vs Predicted')
plt.legend()
plt.show()

mse_test = mean_squared_error(y_true_test, y_pred_test)
r2_test = r2_score(y_true_test, y_pred_test)
print(f"Test Set Mean Squared Error: {mse_test:.4f}")
print(f"Test Set R^2 Score: {r2_test:.4f}")


In [ ]:
# Use the correct source for true values
y_true_train = train_output_df3_LTQOpos_copy['log_response'].values
y_pred_train = ChemNet_MLP_train_output_df3_LTQOpos.cpu().numpy().flatten()

plt.figure(figsize=(6,6))
plt.scatter(y_true_train, y_pred_train, alpha=0.5)
plt.plot([y_true_train.min(), y_true_train.max()], [y_true_train.min(), y_true_train.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('Test Set: True vs Predicted log(Response)')
plt.legend()
plt.show()

mse_train = mean_squared_error(y_true_train, y_pred_train)
r2_train = r2_score(y_true_train, y_pred_train)
print(f"Train Set Mean Squared Error: {mse_train:.4f}")
print(f"Train Set R^2 Score: {r2_train:.4f}")

### MLP on super test set LTQOpos

In [ ]:
# CHECK FOR BUGS. DOESNT PROPERLY MATCH THE SPECTRA TWIN


# --- ChemNet MLP: Predict and evaluate on super test set (NO retraining) ---

# Prepare X for the super test set (ChemNet embeddings)
embedding_cols = [col for col in super_test_output_df3_LTQOpos_withEPA.columns if col.startswith('Embedding Float')]
X_super = super_test_output_df3_LTQOpos_withEPA[embedding_cols].values
y_super = super_test_output_df3_LTQOpos_withEPA['log_response'].values

# Debug information
print(f"Training data shape: {x_train.shape}")
print(f"Super test set shape: {X_super.shape}")
print(f"Number of embedding columns in super test set: {len(embedding_cols)}")
print(f"MLP model expects input size: {x_train.shape[1]}")
print(f"Super test set has: {X_super.shape[1]} features")

# Check if we need to adjust the feature size
if X_super.shape[1] != x_train.shape[1]:
    print(f"Shape mismatch! Need to adjust from {X_super.shape[1]} to {x_train.shape[1]} features")
    
    # Option 1: Truncate the super test set to match training data size
    if X_super.shape[1] > x_train.shape[1]:
        print("Truncating super test set to match training data size")
        X_super = X_super[:, :x_train.shape[1]]
    
    # Option 2: Pad the super test set if it's smaller (unlikely in this case)
    elif X_super.shape[1] < x_train.shape[1]:
        print("Padding super test set to match training data size")
        padding = np.zeros((X_super.shape[0], x_train.shape[1] - X_super.shape[1]))
        X_super = np.concatenate([X_super, padding], axis=1)

print(f"Final super test set shape: {X_super.shape}")

# Convert to tensor and move to device
x_super_tensor = torch.tensor(X_super, dtype=torch.float32).to(device)

# Predict with the already-trained MLP
MLP_model_df3_LTQOpos.eval()
with torch.no_grad():
    y_pred_super = MLP_model_df3_LTQOpos(x_super_tensor).cpu().numpy().flatten()

# Visualization and metrics
plt.figure(figsize=(6,6))
plt.scatter(y_super, y_pred_super, alpha=0.5)
plt.plot([y_super.min(), y_super.max()], [y_super.min(), y_super.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('MLP Predicted log(Response)')
plt.title('MLP Regression: Super Test Set')
plt.legend()
plt.show()

mse_super = mean_squared_error(y_super, y_pred_super)
r2_super = r2_score(y_super, y_pred_super)
print(f"Super Test Set Mean Squared Error: {mse_super:.4f}")
print(f"Super Test Set R^2 Score: {r2_super:.4f}")

### Random Forest Regressor LTQOpos

In [ ]:
# Prepare X and y for the Random Forest
X_rf = test_output_df3_LTQOpos_copy.iloc[:, :-4].select_dtypes(include=[float, int]).values
y_rf = test_output_df3_LTQOpos_copy['log_response'].values

# Train/test split (here, use all for both train and test for direct comparison, or split as needed)
# For demonstration, train and test on the same set
rf_reg_LTQOpos = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg_LTQOpos.fit(X_rf, y_rf)
y_pred_rf = rf_reg_LTQOpos.predict(X_rf)

# Plotting
plt.figure(figsize=(6,6))
plt.scatter(y_rf, y_pred_rf, alpha=0.5)
plt.plot([y_rf.min(), y_rf.max()], [y_rf.min(), y_rf.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('RF Predicted log(Response)')
plt.title('Random Forest Regression: True vs Predicted')
plt.legend()
plt.show()

# Metrics
mse_rf = mean_squared_error(y_rf, y_pred_rf)
r2_rf = r2_score(y_rf, y_pred_rf)
print(f"Random Forest Mean Squared Error: {mse_rf:.4f}")
print(f"Random Forest R^2 Score: {r2_rf:.4f}")

# Spectra --> ChemNet AND Toxicity Conditional Encoder

### Architecture

In [ ]:
batch_size = 64
epochs=500
lr=0.0001
criterion1=nn.MSELoss()
criterion2=nn.MSELoss()
output_size = 513
num_layers = 8

#%%
# Encoder architecture (With Validation Set)
class Cond_Encoder(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model_condenc(model, train_data, val_data, epochs, learning_rate, criterion1, criterion2, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_embeddings, true_log_tox, _ in train_data:
            batch = batch.to(device)
            true_embeddings = true_embeddings.to(device)
            true_log_tox = true_log_tox.to(device)

            optimizer.zero_grad()
            batch_predicted_combined = model(batch) # Take the first 512 for criterion 1 and the last for criterion 2, look up to make sure i only apply the loss to the subset of the model
            
            # Embedding Loss
            batch_predicted_embeddings = batch_predicted_combined[:, :512] # First 512 columns
            loss1 = criterion1(batch_predicted_embeddings, true_embeddings) # loss1 (embedding loss)
            # Response Loss
            batch_predicted_log_tox = batch_predicted_combined[:, 512:] # Last column
            loss2 = criterion2(batch_predicted_log_tox, true_log_tox) # loss2 (toxicity loss)
            
            total_loss = loss1 + loss2
            total_loss.backward()
            optimizer.step()
            running_loss += total_loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():  # Condense this as we did above for symmetry tho not needed without loss.backward command
            for val_batch, val_true_embeddings, val_true_tox, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_embeddings.to(device)
                val_true_tox = val_true_tox.to(device)

                val_batch_predicted = model(val_batch)
                val_batch_predicted_embeddings = val_batch_predicted[:, :512]

                val_loss = criterion1(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss += loss1.item()

                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_tox.to(device)

                val_batch_predicted_tox = val_batch_predicted[:, 512:]

                val_loss = criterion2(val_batch_predicted_tox, val_true_tox)
                val_loss += loss2.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model
#%%


### Training and Testing set construction

In [ ]:
# Training and validation dataset split we need to add in response to the spectra.
df3_QQpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")

In [ ]:
# Add the 'Response' and 'log_response' columns to df3_QQpos_spectra
def add_response_and_log_response(spectra_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds 'Response' and 'log_response' columns to spectra_df by mapping from original_df using the SMILES column.
    """
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    spectra_df['Response'] = spectra_df[smiles_col].map(smiles_to_response)
    spectra_df['log_response'] = np.log(spectra_df['Response'])
    return spectra_df

df3_QQpos_spectra = add_response_and_log_response(df3_QQpos_spectra, df3_QQpos, smiles_col='SMILES_spectra')

In [ ]:
df3_QQpos_spectra.head()

In [ ]:
# Generalize the syntax
dataset = df3_QQpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQpos_copy = train_data.copy()
test_data_df3_QQpos_copy = test_data.copy()

train_data_df3_QQpos = add_response_and_log_response(train_data_df3_QQpos_copy, df3_QQpos, smiles_col='SMILES_spectra')
test_data_df3_QQpos = add_response_and_log_response(test_data_df3_QQpos_copy, df3_QQpos, smiles_col='SMILES_spectra')

# Make a copy
train_data = train_data_df3_QQpos
test_data = test_data_df3_QQpos


# Load val_data
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 

In [ ]:
smiles_to_remove = ['CCCSP(=O)(OCC)SCCC', 'C#CCN(C)Cc1ccccc1', 'C1=CC(=CC=N1)C1=CC=NC=C1', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'C(NC1=NC=NC2=C1N=CN2C1CCCCO1)C1=CC=CC=C1', 'CC(C)N(CC1=CC=CC=C1)C(=O)C(C)(C)C', 'NC(=S)NC1=CC=CC=C1', 
                    'CC(C)(C)C1=CC(=CC(=C1O)C(C)(C)C)[N+]([O-])=O', 'CC(C)(C)C1=CC=C(C=C1)C(O)=O', 'C(NC1=C2N=CN=C2N=CN1)C1=CC=CO1', 
                    'CC(=C)C(=O)NC1=CC=C(Cl)C(Cl)=C1', 'CC(C)(C1=CC(Cl)=C(O)C(Cl)=C1)C1=CC(Cl)=C(O)C(Cl)=C1', 
                    'CCC(=O)N(C1CCN(CCC2=CC=CC=C2)CC1)C1=CC=CC=C1', 'CC(=O)N1CCN(CC1)C1=CC=C(OC[C@H]2CO[C@@](CN3C=CN=C3)(O2)C2=CC=C(Cl)C=C2Cl)C=C1', 
                    'CC(C)(C)C(=O)C(OC1=CC=C(Cl)C=C1)N1C=CN=C1', 'C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1', 'CC(=O)OCC[N+](C)(C)C', 
                    'CC(=O)NC1=CC=C(C=C1)S(=O)(=O)NC1=NOC(C)=C1', 'C1CCC(CC1)NC1CCCCC1', 'CC(CN1CCN(CCOCCO)CC1)CN1C2=CC=CC=C2SC2=CC=CC=C12', 
                    'CC(=O)OCC(COC(C)=O)OC(C)=O', 'CC(C)N(C(=O)CCl)c1ccccc1', 'CC1=C(C)N=C(NS(=O)(=O)C2=CC=C(N)C=C2)O1']

# Combine train and test to extract all rows with those SMILES
combined_df3_QQpos = pd.concat([train_data_df3_QQpos, test_data_df3_QQpos], ignore_index=True)
super_testing_df3_QQpos = combined_df3_QQpos[combined_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

# Remove those SMILES from train and test sets
train_data_df3_QQpos = train_data_df3_QQpos[~train_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()
test_data_df3_QQpos = test_data_df3_QQpos[~test_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)].copy()

print("After removal, train shape:", train_data_df3_QQpos.shape)
print("After removal, test shape:", test_data_df3_QQpos.shape)
print("Super testing set shape:", super_testing_df3_QQpos.shape)
print("Any left in train?", any(train_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)))
print("Any left in test?", any(test_data_df3_QQpos['SMILES_spectra'].isin(smiles_to_remove)))

In [ ]:
test_data_df3_QQpos.head()

In [ ]:
# Ensure we match the data correctly:
train_data = train_data_df3_QQpos
test_data = test_data_df3_QQpos
val_data = test_data

train_data_df3_QQpos_copy2 = train_data.copy()
test_data_df3_QQpos_copy2 = test_data.copy()

In [ ]:
def create_dataset_tensors_emb_tox(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):

    spectra = spectra_dataset.iloc[:,1:-3]

    # create tensors of spectra, true embeddings, true toxicity values, and chemical name encodings for train and val
    chem_labels = list(spectra_dataset['SMILES_spectra'])
    log_tox_tensor = torch.Tensor(spectra_dataset["log_response"].values).unsqueeze(1).to(device)
    embeddings_tensor = torch.Tensor([embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) for chem_name in chem_labels]).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return embeddings_tensor, log_tox_tensor, spectra_tensor, spectra_indices_tensor 

In [ ]:
# MLP training
device = f.set_up_gpu()
# Training set
y_train_emb, y_train_tox, x_train, train_indices_tensor = create_dataset_tensors_emb_tox(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del train_data

# Validation set
y_val_emb, y_val_tox, x_val, val_indices_tensor = create_dataset_tensors_emb_tox(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train_emb, y_train_tox, train_indices_tensor)
val_data = TensorDataset(x_val, y_val_emb, y_val_tox, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
cond_encoder_df3_QQpos = Cond_Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
cond_encoder_model_df3_QQpos = train_model_condenc(
    model=cond_encoder_df3_QQpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion1=criterion1,
    criterion2=criterion2,
    device=device
)

In [ ]:
# Generate the output of the conditional encoder for visulatization and evaulation
# Generate output from the conditional encoder
cond_encoder_model_df3_QQpos.eval()
with torch.no_grad():
    cond_encoder_output = cond_encoder_model_df3_QQpos(x_val)

# Separate the output into ChemNet embeddings (first 512) and toxicity prediction (last 1)
chemnet_output = cond_encoder_output[:, :512].cpu().numpy()  # First 512 dimensions
tox_output = cond_encoder_output[:, 512:].cpu().numpy().flatten()  # Last dimension

print(f"ChemNet output shape: {chemnet_output.shape}")
print(f"Toxicity output shape: {tox_output.shape}")

# True ChemNet embeddings 
true_chemnet_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")
true_chemnet_embeddings = true_chemnet_df.iloc[:, 1:513].values  # Assuming columns 1-512 are embeddings
true_smiles = true_chemnet_df['SMILES'].values  # Adjust column name if different


In [ ]:
# PCA on ChemNet embeddings
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

# Fit PCA on generated embeddings and transform both generated and true embeddings
chemnet_pca = pca.fit_transform(chemnet_output)
true_chemnet_pca = pca.transform(true_chemnet_embeddings)

# Get SMILES for coloring (from your validation data)
val_smiles = test_data_df3_QQpos_copy2['SMILES_spectra'].values  # Adjust if needed

# Create PCA visualization
plt.figure(figsize=(10, 8))

# Plot generated embeddings as circles, colored by SMILES
unique_smiles = np.unique(val_smiles)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_smiles)))

for i, smiles in enumerate(unique_smiles):
    mask = val_smiles == smiles
    plt.scatter(chemnet_pca[mask, 0], chemnet_pca[mask, 1], 
               c=[colors[i]], alpha=0.7, s=50)

# Plot true embeddings as squares
for i, smiles in enumerate(true_smiles):
    if smiles in unique_smiles:
        color_idx = np.where(unique_smiles == smiles)[0][0]
        plt.scatter(true_chemnet_pca[i, 0], true_chemnet_pca[i, 1], 
                   c=[colors[color_idx]], marker='s', s=100, 
                   edgecolors='black', linewidths=1, alpha=0.8)

# Add legend with just the marker types
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
                         markersize=8, label='Generated ChemNet', alpha=0.7),
                  Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', 
                         markersize=8, label='True ChemNet', markeredgecolor='black')]

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA of ChemNet Embeddings: Generated vs True')
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()
# # Show explained variance
# print(f"PCA Explained Variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")
# print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Toxicity prediction accuracy visualization
y_true_tox = y_val_tox.cpu().numpy().flatten()  # True toxicity values
y_pred_tox = tox_output  # Predicted toxicity values

plt.figure(figsize=(6, 6))
plt.scatter(y_true_tox, y_pred_tox, alpha=0.5)
plt.plot([y_true_tox.min(), y_true_tox.max()], [y_true_tox.min(), y_true_tox.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('Conditional Encoder: Toxicity Prediction Accuracy')
plt.legend()
plt.show()

# Calculate metrics for toxicity prediction
from sklearn.metrics import mean_squared_error, r2_score
mse_tox = mean_squared_error(y_true_tox, y_pred_tox)
r2_tox = r2_score(y_true_tox, y_pred_tox)
print(f"Toxicity Prediction MSE: {mse_tox:.4f}")
print(f"Toxicity Prediction R²: {r2_tox:.4f}")

In [ ]:
# Filter the spectra by the minimum number of spectra per SMILES
min_spectra_count = 5 

# Get SMILES and count occurrences of each SMILES and filter
val_smiles = test_data_df3_QQpos_copy2['SMILES_spectra'].values  # Adjust if needed
smiles_counts = pd.Series(val_smiles).value_counts()
frequent_smiles = smiles_counts[smiles_counts >= min_spectra_count].index.tolist()

print(f"Total unique SMILES: {len(smiles_counts)}")
print(f"SMILES with >= {min_spectra_count} spectra: {len(frequent_smiles)}")

# Filter generated embeddings to only include frequent SMILES
frequent_mask = np.isin(val_smiles, frequent_smiles)
chemnet_output_filtered = chemnet_output[frequent_mask]
val_smiles_filtered = val_smiles[frequent_mask]

# Filter true embeddings to only include frequent SMILES
true_frequent_mask = np.isin(true_smiles, frequent_smiles)
true_chemnet_embeddings_filtered = true_chemnet_embeddings[true_frequent_mask]
true_smiles_filtered = true_smiles[true_frequent_mask]

print(f"Filtered generated embeddings shape: {chemnet_output_filtered.shape}")
print(f"Filtered true embeddings shape: {true_chemnet_embeddings_filtered.shape}")

# PCA on filtered ChemNet embeddings
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

# Fit PCA on filtered generated embeddings and transform both filtered datasets
chemnet_pca = pca.fit_transform(chemnet_output_filtered)
true_chemnet_pca = pca.transform(true_chemnet_embeddings_filtered)

# Create PCA visualization
plt.figure(figsize=(10, 8))
unique_frequent_smiles = np.unique(val_smiles_filtered)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_frequent_smiles)))

for i, smiles in enumerate(unique_frequent_smiles):
    mask = val_smiles_filtered == smiles
    plt.scatter(chemnet_pca[mask, 0], chemnet_pca[mask, 1], 
               c=[colors[i]], alpha=0.7, s=50)

for i, smiles in enumerate(true_smiles_filtered):
    if smiles in unique_frequent_smiles:
        color_idx = np.where(unique_frequent_smiles == smiles)[0][0]
        plt.scatter(true_chemnet_pca[i, 0], true_chemnet_pca[i, 1], 
                   c=[colors[color_idx]], marker='s', s=100, 
                   edgecolors='black', linewidths=1, alpha=0.8)

# Add legend with just the marker types
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
                         markersize=8, label='Generated ChemNet', alpha=0.7),
                  Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', 
                         markersize=8, label='True ChemNet', markeredgecolor='black')]

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title(f'PCA of ChemNet Embeddings: Generated vs True (SMILES with ≥{min_spectra_count} spectra)')
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

# Optional: Show explained variance and filtering stats
print(f"PCA Explained Variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")
print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Included SMILES: {unique_frequent_smiles}")

In [ ]:
# Create tensor from super testing data
x_super_df3_QQpos = super_testing_df3_QQpos.iloc[:, 1:-3].values  # Adjust column range as needed
x_super_tensor_df3_QQpos = torch.tensor(x_super_df3_QQpos, dtype=torch.float32)
# Generate output from the conditional encoder on super test set
x_super_tensor_df3_QQpos = x_super_tensor_df3_QQpos.to(next(cond_encoder_model_df3_QQpos.parameters()).device)
cond_encoder_model_df3_QQpos.eval()
with torch.no_grad():
    super_cond_encoder_output = cond_encoder_model_df3_QQpos(x_super_tensor_df3_QQpos)

# Separate the output into ChemNet embeddings (first 512) and toxicity prediction (last 1)
super_chemnet_output = super_cond_encoder_output[:, :512].cpu().numpy()  # First 512 dimensions
super_tox_output = super_cond_encoder_output[:, 512:].cpu().numpy().flatten()  # Last dimension

print(f"Super test ChemNet output shape: {super_chemnet_output.shape}")
print(f"Super test toxicity output shape: {super_tox_output.shape}")

# Load true ChemNet embeddings for comparison
true_chemnet_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")
true_chemnet_embeddings = true_chemnet_df.iloc[:, 1:513].values  # Assuming columns 1-512 are embeddings
true_smiles = true_chemnet_df['SMILES'].values  # Adjust column name if different


In [ ]:
# Filter as before, but in this case I'll want all of them
min_spectra_count = 1  # EASILY CHANGEABLE VALUE

# Get SMILES for super test set
super_smiles = super_testing_df3_QQpos['SMILES_spectra'].values

# Count occurrences and filter (using original validation data for counting)
val_smiles = test_data_df3_QQpos_copy['SMILES_spectra'].values
smiles_counts = pd.Series(val_smiles).value_counts()
frequent_smiles = smiles_counts[smiles_counts >= min_spectra_count].index.tolist()

# Filter super test embeddings to only include frequent SMILES
super_frequent_mask = np.isin(super_smiles, frequent_smiles)
super_chemnet_output_filtered = super_chemnet_output[super_frequent_mask]
super_smiles_filtered = super_smiles[super_frequent_mask]

# Filter true embeddings to only include frequent SMILES that are also in super test
super_relevant_smiles = np.intersect1d(frequent_smiles, super_smiles)
true_super_mask = np.isin(true_smiles, super_relevant_smiles)
true_chemnet_embeddings_filtered = true_chemnet_embeddings[true_super_mask]
true_smiles_filtered = true_smiles[true_super_mask]

print(f"Super test frequent SMILES: {len(np.unique(super_smiles_filtered))}")
print(f"Filtered super test embeddings shape: {super_chemnet_output_filtered.shape}")
print(f"Filtered true embeddings shape: {true_chemnet_embeddings_filtered.shape}")

# PCA on filtered ChemNet embeddings
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

# Fit PCA on filtered super test embeddings and transform both datasets
super_chemnet_pca = pca.fit_transform(super_chemnet_output_filtered)
true_chemnet_pca = pca.transform(true_chemnet_embeddings_filtered)

# Create PCA visualization
plt.figure(figsize=(10, 8))

# Plot generated embeddings as circles, colored by SMILES
unique_super_smiles = np.unique(super_smiles_filtered)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_super_smiles)))

for i, smiles in enumerate(unique_super_smiles):
    mask = super_smiles_filtered == smiles
    plt.scatter(super_chemnet_pca[mask, 0], super_chemnet_pca[mask, 1], 
               c=[colors[i]], alpha=0.7, s=50)

# Plot true embeddings as squares
for i, smiles in enumerate(true_smiles_filtered):
    if smiles in unique_super_smiles:
        color_idx = np.where(unique_super_smiles == smiles)[0][0]
        plt.scatter(true_chemnet_pca[i, 0], true_chemnet_pca[i, 1], 
                   c=[colors[color_idx]], marker='s', s=100, 
                   edgecolors='black', linewidths=1, alpha=0.8)

# Add legend with just the marker types
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
                         markersize=8, label='Generated ChemNet', alpha=0.7),
                  Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', 
                         markersize=8, label='True ChemNet', markeredgecolor='black')]

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title(f'PCA of ChemNet Embeddings - Super Test Set (SMILES with ≥{min_spectra_count} spectra)')
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()



# # Show explained variance
# print(f"PCA Explained Variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")
# print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Toxicity prediction accuracy visualization for super test set
y_true_super_tox = super_testing_df3_QQpos['log_response'].values
y_pred_super_tox = super_tox_output

# Align arrays in case of size mismatch
min_length = min(len(y_true_super_tox), len(y_pred_super_tox))
y_true_super_tox = y_true_super_tox[:min_length]
y_pred_super_tox = y_pred_super_tox[:min_length]

plt.figure(figsize=(6, 6))
plt.scatter(y_true_super_tox, y_pred_super_tox, alpha=0.5)
plt.plot([y_true_super_tox.min(), y_true_super_tox.max()], [y_true_super_tox.min(), y_true_super_tox.max()], 'r--', label='Ideal')
plt.xlabel('True log(Response)')
plt.ylabel('Predicted log(Response)')
plt.title('Conditional Encoder: Super Test Set Toxicity Prediction')
plt.legend()
plt.show()

# Calculate metrics for super test toxicity prediction
from sklearn.metrics import mean_squared_error, r2_score
mse_super_tox = mean_squared_error(y_true_super_tox, y_pred_super_tox)
r2_super_tox = r2_score(y_true_super_tox, y_pred_super_tox)
print(f"Super Test Toxicity Prediction MSE: {mse_super_tox:.4f}")
print(f"Super Test Toxicity Prediction R²: {r2_super_tox:.4f}")